**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure3_regulons')


In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns


from scipy.spatial.distance import jensenshannon

# ── RSS 계산 ─────────────────────────────────────────────────
# RSS: 각 TF의 AUC 분포가 특정 celltype의 cell 분포와
#      얼마나 일치하는지 (Jensen-Shannon divergence 기반)
# RSS 높을수록 해당 celltype에 특이적인 regulon

def calc_rss(auc_df, cell_type_series):
    celltypes = cell_type_series.unique()
    rss = {ct: {} for ct in celltypes}
    for ct in celltypes:
        # 해당 celltype에 속하는지 0/1 벡터 → 정규화
        mask = (cell_type_series == ct).astype(float)
        p = mask / mask.sum()
        for tf in auc_df.columns:
            q = auc_df[tf].values
            q = q / q.sum() if q.sum() > 0 else q
            rss[ct][tf] = 1 - jensenshannon(p, q)
    return pd.DataFrame(rss)   # index: TF, columns: celltype

In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

# pySCENIC analysis

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))
adata

In [ ]:
print(adata)  # AnnData object with n_obs × n_vars = ?
print(adata.X.__class__)  # numpy.ndarray or scipy.sparse.csr_matrix?

In [ ]:
scenic_adata = adata.copy()  # Preserve annotations/UMAP for later Figure3 cells.
#pre process scenic_adata to analyze with SCENIC
import numpy as np

print(scenic_adata.X.shape)

scenic_adata.layers = {}
scenic_adata.obsm = {}
scenic_adata.varm = {}
scenic_adata.obsp = {}

scenic_adata.obs = scenic_adata.obs[["n_counts"]] 
scenic_adata.var = scenic_adata.var[[]]

scenic_adata.var_names_make_unique()  
scenic_adata.var["Gene"] = scenic_adata.var_names
scenic_adata.obs["CellID"] = scenic_adata.obs.index
scenic_adata.write_loom(output_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/scRNA_adata_final.loom'), write_obsm_varm=False)

import loompy

with loompy.connect(output_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/scRNA_adata_final.loom'), mode="r") as ds:
    print("Row attributes:", ds.ra.keys())  
    print("Column attributes:", ds.ca.keys())  

pySCENIC `grn` 단계는 저장소 루트에서 실행합니다.

```bash
bash scripts/external/pyscenic.sh grn
```


pySCENIC `ctx` 단계는 저장소 루트에서 실행합니다.

```bash
bash scripts/external/pyscenic.sh ctx
```


pySCENIC `aucell` 단계는 저장소 루트에서 실행합니다.

```bash
bash scripts/external/pyscenic.sh aucell
```


In [ ]:
adj_matrix = pd.read_csv(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv'), sep='\t', index_col=0)
motif_matrix = pd.read_csv(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.motifs.csv'))
auc_matrix = pd.read_csv(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.auc.csv'), index_col=0)

In [ ]:
adata.obs = adata.obs.join(auc_matrix, how='left')
print(adata.obs)

In [ ]:
assert all(auc_matrix.index.isin(adata.obs.index)), "AUC index와 adata.obs index가 일치하지 않습니다."

In [ ]:
filtered_matrix = motif_matrix.iloc[:, [0, 8]]
filtered_matrix

In [ ]:
import ast 
filtered_matrix = motif_matrix.iloc[:, [0, 8]]
filtered_matrix = filtered_matrix.iloc[2:].reset_index(drop=True)
filtered_matrix.columns = ['TF', 'TargetGenes']
filtered_matrix['TargetGenes'] = filtered_matrix['TargetGenes'].apply(
 lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
filtered_matrix['TargetGenes'] = filtered_matrix['TargetGenes'].apply(
 lambda x: [gene[0] for gene in x])
regulons = filtered_matrix.groupby('TF')['TargetGenes'].apply(lambda x: sum(x, [])).to_dict()

In [ ]:
auc_data = adata.obs.loc[:, adata.obs.columns.str.endswith('(+)')]
clusters = adata.obs['celltype']
clustered_auc = auc_data.groupby(clusters).mean()

In [ ]:
clustered_auc

In [ ]:
# ── 1. TF-Target edge list 생성 ───────────────────────────────
edge_rows = []
for tf, targets in regulon_sp.items():
    group = tf_to_group.get(tf, 'Unknown')
    for gene in targets:
        edge_rows.append({'TF': tf, 'Target': gene, 'TF_Group': group})

edge_df = pd.DataFrame(edge_rows)
print(f'Total TF-Target edges (from regulon_sp): {len(edge_df)}')

# ── 2. adj importance 합치기 ──────────────────────────────────
adj = pd.read_csv(
    artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv'),
    sep='\t'
)
adj_sp = adj[adj['TF'].isin(SPECIAL_TFS)].copy()
adj_sp.columns = ['TF', 'Target', 'importance']

edge_df = edge_df.merge(adj_sp[['TF', 'Target', 'importance']],
                         on=['TF', 'Target'], how='left')
print(f'importance matched: {edge_df["importance"].notna().sum()} / {len(edge_df)}')

# ── 3. NES (TF별 최고값) 합치기 ──────────────────────────────
motif_matrix_raw = pd.read_csv(
    artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.motifs.csv'),
    header=[0, 1], index_col=[0, 1]
)
# MultiIndex Series → reset → 컬럼명 명시적 지정
nes_raw = motif_matrix_raw[('Enrichment', 'NES')].reset_index()
nes_raw.columns = ['TF', 'MotifID', 'NES']
nes_df = nes_raw.groupby('TF')['NES'].max().reset_index()  # TF당 최고 NES

edge_df = edge_df.merge(nes_df, on='TF', how='left')
print(f'NES matched: {edge_df["NES"].notna().sum()} / {len(edge_df)}')

# ── 4. TF 메타데이터 (log2FC, z scores) 합치기 ───────────────
df_clean = df.copy().reset_index(drop=True)
df_clean['TF'] = df.index.str.replace('(+)', '', regex=False)
zdf_reset = zdf.reset_index(drop=True)
tf_meta = df_clean[['TF', 'log2FC', 'padj', 'mean_Classical', 'mean_SLC7A5_high']].copy()
tf_meta = tf_meta.merge(
    zdf_reset[['TF_clean', 'z_classical', 'z_special', 'delta_z']]
    .rename(columns={'TF_clean': 'TF'}),
    on='TF', how='left'
)

edge_df = edge_df.merge(tf_meta, on='TF', how='left')

# ── 5. 컬럼 정리 & 정렬 ──────────────────────────────────────
edge_df = edge_df[[
    'TF_Group', 'TF', 'Target', 'importance',
    'NES', 'log2FC', 'padj',
    'z_special', 'z_classical', 'delta_z',
    'mean_Classical', 'mean_SLC7A5_high'
]].sort_values(['TF_Group', 'TF', 'importance'], ascending=[True, True, False])

edge_df = edge_df.round({
    'importance': 4, 'NES': 4, 'log2FC': 4, 'padj': 6,
    'z_special': 4, 'z_classical': 4, 'delta_z': 4,
    'mean_Classical': 6, 'mean_SLC7A5_high': 6
})

print(f'\nFinal edge table shape: {edge_df.shape}')
print(edge_df.head(5).to_string(index=False))

# ── 6. TF 노드 테이블 ─────────────────────────────────────────
tf_node_df = (
    edge_df[['TF_Group', 'TF', 'NES', 'log2FC', 'padj',
              'z_special', 'z_classical', 'delta_z',
              'mean_Classical', 'mean_SLC7A5_high']]
    .drop_duplicates(subset='TF')
    .sort_values(['TF_Group', 'TF'])
)
tf_node_df['n_targets'] = tf_node_df['TF'].map(
    edge_df.groupby('TF')['Target'].count()
)

# ── 7. Target 노드 테이블 ────────────────────────────────────
target_node_df = (
    edge_df.groupby('Target')
    .agg(
        n_TFs=('TF', 'nunique'),
        TFs_targeting=('TF', lambda x: ', '.join(sorted(x.unique()))),
        Groups_targeting=('TF_Group', lambda x: ', '.join(sorted(x.unique()))),
        mean_importance=('importance', 'mean'),
        max_importance=('importance', 'max'),
    )
    .reset_index()
    .sort_values('n_TFs', ascending=False)
)
target_node_df['is_hub'] = target_node_df['n_TFs'] >= 3
target_node_df = target_node_df.round({'mean_importance': 4, 'max_importance': 4})

print(f'\nTF nodes    : {len(tf_node_df)}')
print(f'Target nodes: {len(target_node_df)}')
print(f'Hub genes (n_TFs >= 3): {target_node_df["is_hub"].sum()}')
print('\nTop 10 hub genes:')
print(target_node_df.head(10)[['Target','n_TFs','TFs_targeting','mean_importance']].to_string(index=False))

# ── 8. Excel 저장 ────────────────────────────────────────────
save_xlsx = output_path('02_SLC7A5_mono_Journal/04_Table/Special_monocyte_regulon.xlsx')

with pd.ExcelWriter(save_xlsx, engine='openpyxl') as writer:
    edge_df.to_excel(writer, sheet_name='Edge_list', index=False)
    tf_node_df.to_excel(writer, sheet_name='TF_nodes', index=False)
    target_node_df.to_excel(writer, sheet_name='Target_nodes', index=False)

print(f'\nSaved: {save_xlsx}')
print('Sheets: Edge_list | TF_nodes | Target_nodes')


# Classical monocyte TF anlaysis

In [ ]:
#Classical_monocyte TF
cluster_name = "Classical_monocyte"
clusters = adata.obs['celltype']

adata.obs['0vs'] = clusters.apply(lambda x: x if x == 'Classical_monocyte' else 'other')
cluster_auc = clustered_auc.loc[cluster_name]
top_20_tfs = cluster_auc.sort_values(ascending=False).head(20)
top_20_tfs.replace("(+)", "")
# 결과 출력
print(top_20_tfs)

In [ ]:
grouped_auc = auc_data.groupby(adata.obs['0vs']).mean()
grouped_auc

In [ ]:
# t-test

In [ ]:
from scipy.stats import ttest_ind

# 1번 클러스터와 other 그룹으로 데이터를 나누기
group_1 = auc_data[adata.obs['0vs'] == 'Classical_monocyte']
group_other = auc_data[adata.obs['0vs'] == 'other']

# t-test 수행
p_values = {}
for tf in auc_data.columns:  # 각 TF에 대해 수행
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

# 결과를 정리하여 데이터프레임으로 변환
import pandas as pd
results = pd.DataFrame({
    'TF': p_values.keys(),
    'p-value': p_values.values()
}).set_index('TF')

# p-value로 정렬
results = results.sort_values('p-value')
print(results)

In [ ]:
from statsmodels.stats.multitest import multipletests
# 1. t-test로 p-value 계산
p_values = {}
z_scores = {}
for tf in auc_data.columns:
    # t-test
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

    # z-score
    mean_1, mean_other = group_1[tf].mean(), group_other[tf].mean()
    std_1, std_other = group_1[tf].std(), group_other[tf].std()
    z_scores[tf] = (mean_1 - mean_other) / np.sqrt((std_1**2 / len(group_1)) + (std_other**2 / len(group_other)))
    # p-values와 z-scores를 데이터프레임으로 정리
results = pd.DataFrame({
    'TF': auc_data.columns,
    'p-value': [p_values[tf] for tf in auc_data.columns],
    'z-score': [z_scores[tf] for tf in auc_data.columns]
}).set_index('TF')

# 2. FDR 보정
reject, pvals_corrected, _, _ = multipletests(results['p-value'], method='fdr_bh')
results['adjusted p-value'] = pvals_corrected
results['significant'] = reject  # True/False

# 3. -log10(adjusted p-value) 계산
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'])
results

In [ ]:
# z-score 기준으로 내림차순 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 유의미한 TF만 필터링 (adjusted p-value < 0.05)
sig_results = results_sorted[results_sorted['significant'] == True]

# 상위 20개만 보기
top_sig = sig_results.head(20)
top_sig

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top_sig.index[::-1], top_sig['z-score'][::-1])
plt.xlabel('Z-score')
plt.title('Top 20 Significant TFs (mono1)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Z-score 기준 상위 및 하위 6개 TF 선택
top_positive = results.nlargest(6, 'z-score')
top_negative = results.nsmallest(6, 'z-score')

# Volcano plot 생성
plt.figure(figsize=(10, 6))

# 유의미한 TF와 비유의미한 TF 분리
significant = results[results['significant']]
not_significant = results[~results['significant']]

# Volcano plot 그리기
plt.scatter(not_significant['z-score'], not_significant['-log10(adj p-value)'], color='grey', alpha=0.7, label='Not Significant')
plt.scatter(significant['z-score'], significant['-log10(adj p-value)'], color='red', alpha=0.7, label='Significant')

# z-score 기준 상위 및 하위 6개 TF 레이블 추가
for tf in top_positive.index:
    plt.text(
        top_positive.loc[tf, 'z-score'],
        top_positive.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='blue'
    )

for tf in top_negative.index:
    plt.text(
        top_negative.loc[tf, 'z-score'],
        top_negative.loc[tf, '-log10(adj p-value)'],
        tf,
        fontsize=8,
        color='green'
    )

# 가이드라인 추가 (예: p-value 및 z-score 기준선)
plt.axhline(-np.log10(0.05), color='blue', linestyle='--', label='p = 0.05')
plt.axvline(-1, color='green', linestyle='--', label='z = -1')
plt.axvline(1, color='green', linestyle='--', label='z = 1')

# 축 레이블 및 제목
plt.xlabel('Z-score (Cluster 1 vs Other)', fontsize=12)
plt.ylabel('-log10(Adjusted p-value)', fontsize=12)
plt.title('Volcano Plot of TF AUC Scores', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

# Plot 표시
plt.show()

# 상위 및 하위 5개 TF 출력
print("Top 6 TFs with highest z-scores:")
print(top_positive)

print("\nTop 6 TFs with lowest z-scores:")
print(top_negative)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 무한 방지용 epsilon
epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

# z-score 기준 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 상위 3개 TF
top5 = results_sorted.head(5)
print("Top 5 TFs in mono1 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

# 상위 3개 강조
for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in mono1 cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 텍스트 라벨 추가
for tf in top5.index:
    y = results_sorted.loc[tf, 'z-score']
    plt.text(tf, y + 0.2, f"{tf}\n({y:.2f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 무한 방지용 epsilon
epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

# z-score 기준 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 상위 3개 TF
top5 = results_sorted.head(5)
print("Top 5 TFs in mono1 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

# 상위 3개 강조
for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in Special mono cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 텍스트 라벨 추가
for tf in top5.index:
    y = results_sorted.loc[tf, 'z-score']
    plt.text(tf, y + 0.2, f"{tf}\n({y:.2f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3a_1_v1.pdf'), bbox_inches='tight')

In [ ]:

# Classical_monocyte 세포들만
auc_mono = auc_data[adata.obs['celltype'] == 'Classical_monocyte']

# 유의한 TF만 선택
sig_tfs = sig_results.index.tolist()  # Cell 18에서 만든 것                                                                               
auc_sig = auc_mono[sig_tfs]                                                                                                               

# Clustermap                                                                                                                              
sns.clustermap(                                                                                                                           
    auc_sig.T.corr(),  # TF 간 상관관계
    cmap='RdBu_r', center=0,                                                                                                              
    figsize=(10, 10),                                                                                                                     
    xticklabels=True, yticklabels=True                                                                                                    
)                                                                                                                                         
plt.title('TF Co-activity Correlation (Classical_monocyte)')


In [ ]:
# RSS (Regulon Specificity Score) 기반 Classical_monocyte TF 분석

In [ ]:
from scipy.spatial.distance import jensenshannon

# ── RSS 계산 ─────────────────────────────────────────────────
# RSS: 각 TF의 AUC 분포가 특정 celltype의 cell 분포와
#      얼마나 일치하는지 (Jensen-Shannon divergence 기반)
# RSS 높을수록 해당 celltype에 특이적인 regulon

def calc_rss(auc_df, cell_type_series):
    celltypes = cell_type_series.unique()
    rss = {ct: {} for ct in celltypes}
    for ct in celltypes:
        # 해당 celltype에 속하는지 0/1 벡터 → 정규화
        mask = (cell_type_series == ct).astype(float)
        p = mask / mask.sum()
        for tf in auc_df.columns:
            q = auc_df[tf].values
            q = q / q.sum() if q.sum() > 0 else q
            rss[ct][tf] = 1 - jensenshannon(p, q)
    return pd.DataFrame(rss)   # index: TF, columns: celltype

rss_df = calc_rss(auc_data, adata.obs['celltype'])
print(f"RSS matrix shape: {rss_df.shape}  (TF x celltype)")
rss_df.head()

In [ ]:
# Classical_monocyte 특이적 TF — RSS 상위 15개
cluster_name = 'Classical_monocyte'

rss_mono = (rss_df[cluster_name]
            .sort_values(ascending=False)
            .reset_index())
rss_mono.columns = ['TF', 'RSS']
rss_mono['TF_clean'] = rss_mono['TF'].str.replace('(+)', '', regex=False)
print(rss_mono.head(15).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

top_n = 15
top_rss = rss_mono.head(top_n)

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

bars = ax.barh(top_rss['TF_clean'][::-1], top_rss['RSS'][::-1],
               color='#3498DB', edgecolor='white', linewidth=0.5)

# 상위 5개 강조
top5_names = set(top_rss['TF_clean'].head(5))
for bar, tf_name in zip(bars[::-1], top_rss['TF_clean']):
    if tf_name in top5_names:
        bar.set_color('#1A5276')
        bar.set_edgecolor('black')
        bar.set_linewidth(1.0)

ax.set_xlabel('RSS (Regulon Specificity Score)', fontsize=11)
ax.set_title(f'Top {top_n} TFs — {cluster_name}\n(RSS)', fontsize=12)
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# celltype별 top-5 RSS TF를 모아서 union
top_per_ct = set()
for ct in rss_df.columns:
    top_per_ct.update(rss_df[ct].nlargest(5).index.tolist())

# 그 중 Classical_monocyte RSS 기준으로 정렬
top_tfs_ordered = (rss_df.loc[list(top_per_ct), cluster_name]
                   .sort_values(ascending=False).index.tolist())

subset = rss_df.loc[top_tfs_ordered].copy()
subset.index = subset.index.str.replace('(+)', '', regex=False)

plt.figure(figsize=(len(rss_df.columns) * 0.9 + 1, len(top_tfs_ordered) * 0.45 + 1), dpi=150)
sns.set(font_scale=0.75)
sns.heatmap(
    subset,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='lightgrey',
    annot=False,
    cbar_kws={'label': 'RSS', 'shrink': 0.6},
    xticklabels=True,
    yticklabels=True
)
plt.title('Regulon Specificity Score — top TFs per celltype', fontsize=11)
plt.xlabel('Cell Type', fontsize=10)
plt.ylabel('TF (Regulon)', fontsize=10)
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Classical_monocyte RSS vs 나머지 celltype 평균 RSS — scatter
other_cts = [c for c in rss_df.columns if c != cluster_name]
rss_other_mean = rss_df[other_cts].mean(axis=1)

scatter_df = pd.DataFrame({
    'RSS_mono': rss_df[cluster_name],
    'RSS_other_mean': rss_other_mean
})
scatter_df['delta_RSS'] = scatter_df['RSS_mono'] - scatter_df['RSS_other_mean']
scatter_df['TF_clean'] = scatter_df.index.str.replace('(+)', '', regex=False)

# 상위 라벨 기준
top_label = scatter_df.nlargest(10, 'delta_RSS')

sns.reset_defaults()  # 이전 셀 sns.set() 스타일 초기화
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.scatter(scatter_df['RSS_other_mean'], scatter_df['RSS_mono'],
           alpha=0.4, s=15, color='grey', label='All TFs')
ax.scatter(top_label['RSS_other_mean'], top_label['RSS_mono'],
           alpha=0.9, s=40, color='#C0392B', label=f'Top 10 (delta RSS)')

for _, row in top_label.iterrows():
    ax.annotate(row['TF_clean'],
                xy=(row['RSS_other_mean'], row['RSS_mono']),
                xytext=(5, 2), textcoords='offset points',
                fontsize=7.5, color='#1A252F')

# 대각선 (동등한 RSS 기준선)
lim = max(scatter_df[['RSS_mono', 'RSS_other_mean']].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.5, label='y = x')

ax.set_xlabel('Mean RSS (other celltypes)', fontsize=10)
ax.set_ylabel(f'RSS ({cluster_name})', fontsize=10)
ax.set_title('Classical_monocyte-specific regulons\n(above diagonal = more specific)', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 10 Classical_monocyte-specific TFs (by delta RSS):")
print(scatter_df.nlargest(10, 'delta_RSS')[['TF_clean', 'RSS_mono', 'RSS_other_mean', 'delta_RSS']]
      .to_string(index=False))

In [ ]:
from scipy.spatial.distance import jensenshannon                                                                                          

def calc_rss(auc_data, cell_type_series):                                                                                                 
    rss = {}    
    for ct in cell_type_series.unique():                                                                                                  
        mask = (cell_type_series == ct).astype(float)
        p = mask / mask.sum()  # cell type 분포                                                                                           
        rss[ct] = {}                                                                                                                      
        for tf in auc_data.columns:                                                                                                       
            q = auc_data[tf] / auc_data[tf].sum()  # regulon activity 분포                                                                
            rss[ct][tf] = 1 - jensenshannon(p, q)                                                                                         
    return pd.DataFrame(rss)                                                                                                              

rss_df = calc_rss(auc_data, adata.obs['celltype'])                                                                                        
# Classical_monocyte 특이적 TF top 10
rss_df['Classical_monocyte'].sort_values(ascending=False).head(10)  

In [ ]:
import gseapy as gp

# FOSL2 regulon의 target genes 가져오기
top_tf = 'FOSL2'
target_genes = regulons[top_tf]  # Cell 11에서 만든 dict                                                                                  

# Enrichment                                                                                                                              
enr = gp.enrichr(                                                                                                                         
    gene_list=target_genes,                                                                                                               
    gene_sets=['MSigDB_Hallmark_2020', 'KEGG_2021_Human'],
    organism='Human',                                                                                                                     
    outdir=None 
)                                                                                                                                         
enr.results.sort_values('Adjusted P-value').head(10)

In [ ]:
# UMAP이 adata에 있다고 가정
top_tfs = ['FOSL2(+)', 'BACH1(+)', 'FOS(+)', 'BATF(+)']                                                                                   

for tf in top_tfs:                                                                                                                        
    adata.obs[tf] = auc_data[tf].reindex(adata.obs.index)                                                                                 

sc.pl.umap(adata, color=top_tfs, ncols=2, cmap='Reds', vmin=0) 

In [ ]:
sc.pl.umap(adata,color='FOSL2(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,) 
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_FOSL2.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='BACH1(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)  
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_BACH1.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='FOS(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_FOS.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='BATF(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_BATF.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='TGIF1(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_TGIF1.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
# UMAP이 adata에 있다고 가정
top_tfs = ['FOSL2(+)', 'BACH1(+)', 'FOS(+)', 'BATF(+)','TGIF1(+)']                                                                                   

for tf in top_tfs:                                                                                                                        
    adata.obs[tf] = auc_data[tf].reindex(adata.obs.index)                                                                                 

sc.pl.umap(adata, color=top_tfs, ncols=2, cmap='viridis', vmin=0) 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
selected_tfs = ['FOSL2(+)','BACH1(+)', 'FOS(+)', 'BATF(+)','TGIF1(+)', 'SP1(+)', 'FOSB(+)','YY1(+)','ATF4(+)','MYB(+)']  
subset_auc = clustered_auc[selected_tfs]
# Figure 사이즈 및 스타일 지정 (논문용)
plt.figure(figsize=(6, 4), dpi=300)  # 논문용 해상도와 크기
sns.set(font_scale=0.5)              # 글자 크기

# Heatmap
sns.heatmap(
    subset_auc, 
    cmap='bwr',   # 혹은 'coolwarm', 'magma', 'plasma' 등
    linewidths=0.5,   # 셀 간격
    linecolor='grey', # 셀 테두리 색
    annot=True,      # 값 표시 (원하면 True, 아니면 False)
    fmt=".2f",       # 소수점 두 자리
    cbar_kws={"label": "AUC Score"},  # 컬러바 라벨
    square=True      # 정사각형 셀
)

# 축 라벨 조정 (옵션)
plt.xlabel('Transcription Factors', fontsize=12)
plt.ylabel('Cluster', fontsize=12)

# SLC7A5 high monocyte analysis

In [ ]:
adata.obs['celltype']

In [ ]:
#Classical_monocyte TF
cluster_name = "Special_monocyte"
clusters = adata.obs['celltype']

adata.obs['5vs'] = clusters.apply(lambda x: x if x == 'Special_monocyte' else 'other')
cluster_auc = clustered_auc.loc[cluster_name]
top_20_tfs = cluster_auc.sort_values(ascending=False).head(20)
top_20_tfs.replace("(+)", "")
# 결과 출력
print(top_20_tfs)

In [ ]:
grouped_auc = auc_data.groupby(adata.obs['5vs']).mean()
grouped_auc

In [ ]:
from scipy.stats import ttest_ind

# 1번 클러스터와 other 그룹으로 데이터를 나누기
group_1 = auc_data[adata.obs['5vs'] == 'Special_monocyte']
group_other = auc_data[adata.obs['5vs'] == 'other']

# t-test 수행
p_values = {}
for tf in auc_data.columns:  # 각 TF에 대해 수행
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

# 결과를 정리하여 데이터프레임으로 변환
import pandas as pd
results = pd.DataFrame({
    'TF': p_values.keys(),
    'p-value': p_values.values()
}).set_index('TF')

# p-value로 정렬
results = results.sort_values('p-value')
print(results)

In [ ]:
from statsmodels.stats.multitest import multipletests
# 1. t-test로 p-value 계산
p_values = {}
z_scores = {}
for tf in auc_data.columns:
    # t-test
    stat, p_val = ttest_ind(group_1[tf], group_other[tf], equal_var=False)
    p_values[tf] = p_val

    # z-score
    mean_1, mean_other = group_1[tf].mean(), group_other[tf].mean()
    std_1, std_other = group_1[tf].std(), group_other[tf].std()
    z_scores[tf] = (mean_1 - mean_other) / np.sqrt((std_1**2 / len(group_1)) + (std_other**2 / len(group_other)))
    # p-values와 z-scores를 데이터프레임으로 정리
results = pd.DataFrame({
    'TF': auc_data.columns,
    'p-value': [p_values[tf] for tf in auc_data.columns],
    'z-score': [z_scores[tf] for tf in auc_data.columns]
}).set_index('TF')

# 2. FDR 보정
reject, pvals_corrected, _, _ = multipletests(results['p-value'], method='fdr_bh')
results['adjusted p-value'] = pvals_corrected
results['significant'] = reject  # True/False

# 3. -log10(adjusted p-value) 계산
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'])
results

In [ ]:
# z-score 기준으로 내림차순 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 유의미한 TF만 필터링 (adjusted p-value < 0.05)
sig_results = results_sorted[results_sorted['significant'] == True]

# 상위 20개만 보기
top_sig = sig_results.head(20)
top_sig

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(top_sig.index[::-1], top_sig['z-score'][::-1])
plt.xlabel('Z-score')
plt.title('Top 20 Significant TFs (mono1)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 무한 방지용 epsilon
epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

# z-score 기준 정렬
results_sorted = results.sort_values(by='z-score', ascending=False)

# 상위 3개 TF
top5 = results_sorted.head(5)
print("Top 5 TFs in mono1 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

# 상위 3개 강조
for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in Special mono cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 텍스트 라벨 추가
for tf in top5.index:
    y = results_sorted.loc[tf, 'z-score']
    plt.text(tf, y + 0.2, f"{tf}\n({y:.2f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3a_2_v1.pdf'), bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

epsilon = np.nextafter(0, 1)
results['-log10(adj p-value)'] = -np.log10(results['adjusted p-value'] + epsilon)

results_sorted = results.sort_values(by='z-score', ascending=False)

top5 = results_sorted.head(5)
print("Top 5 TFs in mono1 cluster:")
print(top5)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_sorted.index[:20], results_sorted['z-score'][:20], alpha=0.6)

for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        bar.set_alpha(1.0)
        bar.set_edgecolor('black')
        bar.set_linewidth(1.5)

plt.xticks(rotation=75, ha='right')
plt.ylabel('Z-score')
plt.title('Top TFs in Special mono cluster (highlighted top 5)')
plt.grid(axis='y', linestyle='--', alpha=0.5)

for i, bar in enumerate(bars):
    tf_name = results_sorted.index[i]
    if tf_name in top5.index:
        height = bar.get_height()
        x = bar.get_x() + bar.get_width() / 2
        y = height - 0.3
        plt.text(
            x,
            y,
            f"{tf_name}\n({height:.2f})",
            ha='center',
            va='top',
            fontsize=9,
            fontweight='bold',
            color='white'
        )

plt.tight_layout()
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3a_2_v1.1.pdf'),
    bbox_inches='tight'
)

In [ ]:
from scipy.spatial.distance import jensenshannon

# ── RSS 계산 ─────────────────────────────────────────────────
# RSS: 각 TF의 AUC 분포가 특정 celltype의 cell 분포와
#      얼마나 일치하는지 (Jensen-Shannon divergence 기반)
# RSS 높을수록 해당 celltype에 특이적인 regulon

def calc_rss(auc_df, cell_type_series):
    celltypes = cell_type_series.unique()
    rss = {ct: {} for ct in celltypes}
    for ct in celltypes:
        # 해당 celltype에 속하는지 0/1 벡터 → 정규화
        mask = (cell_type_series == ct).astype(float)
        p = mask / mask.sum()
        for tf in auc_df.columns:
            q = auc_df[tf].values
            q = q / q.sum() if q.sum() > 0 else q
            rss[ct][tf] = 1 - jensenshannon(p, q)
    return pd.DataFrame(rss)   # index: TF, columns: celltype

rss_df = calc_rss(auc_data, adata.obs['celltype'])
print(f"RSS matrix shape: {rss_df.shape}  (TF x celltype)")
rss_df.head()

In [ ]:
# Classical_monocyte 특이적 TF — RSS 상위 15개
cluster_name = 'Special_monocyte'

rss_mono = (rss_df[cluster_name]
            .sort_values(ascending=False)
            .reset_index())
rss_mono.columns = ['TF', 'RSS']
rss_mono['TF_clean'] = rss_mono['TF'].str.replace('(+)', '', regex=False)
print(rss_mono.head(15).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

top_n = 15
top_rss = rss_mono.head(top_n)

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

bars = ax.barh(top_rss['TF_clean'][::-1], top_rss['RSS'][::-1],
               color='#3498DB', edgecolor='white', linewidth=0.5)

# 상위 5개 강조
top5_names = set(top_rss['TF_clean'].head(5))
for bar, tf_name in zip(bars[::-1], top_rss['TF_clean']):
    if tf_name in top5_names:
        bar.set_color('#1A5276')
        bar.set_edgecolor('black')
        bar.set_linewidth(1.0)

ax.set_xlabel('RSS (Regulon Specificity Score)', fontsize=11)
ax.set_title(f'Top {top_n} TFs — {cluster_name}\n(RSS)', fontsize=12)
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3a_2_v2.pdf'), bbox_inches='tight')

In [ ]:
#Special_monocyte RSS vs 나머지 celltype 평균 RSS — scatter
other_cts = [c for c in rss_df.columns if c != cluster_name]
rss_other_mean = rss_df[other_cts].mean(axis=1)

scatter_df = pd.DataFrame({
    'RSS_mono': rss_df[cluster_name],
    'RSS_other_mean': rss_other_mean
})
scatter_df['delta_RSS'] = scatter_df['RSS_mono'] - scatter_df['RSS_other_mean']
scatter_df['TF_clean'] = scatter_df.index.str.replace('(+)', '', regex=False)

# 상위 라벨 기준
top_label = scatter_df.nlargest(10, 'delta_RSS')

sns.reset_defaults()  # 이전 셀 sns.set() 스타일 초기화
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.scatter(scatter_df['RSS_other_mean'], scatter_df['RSS_mono'],
           alpha=0.4, s=15, color='grey', label='All TFs')
ax.scatter(top_label['RSS_other_mean'], top_label['RSS_mono'],
           alpha=0.9, s=40, color='#C0392B', label=f'Top 10 (delta RSS)')

for _, row in top_label.iterrows():
    ax.annotate(row['TF_clean'],
                xy=(row['RSS_other_mean'], row['RSS_mono']),
                xytext=(5, 2), textcoords='offset points',
                fontsize=7.5, color='#1A252F')

# 대각선 (동등한 RSS 기준선)
lim = max(scatter_df[['RSS_mono', 'RSS_other_mean']].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.5, label='y = x')

ax.set_xlabel('Mean RSS (other celltypes)', fontsize=10)
ax.set_ylabel(f'RSS ({cluster_name})', fontsize=10)
ax.set_title('Special_monocyte-specific regulons\n(above diagonal = more specific)', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 10 Special_monocyte-specific TFs (by delta RSS):")
print(scatter_df.nlargest(10, 'delta_RSS')[['TF_clean', 'RSS_mono', 'RSS_other_mean', 'delta_RSS']]
      .to_string(index=False))

In [ ]:
from scipy.spatial.distance import jensenshannon                                                                                          

def calc_rss(auc_data, cell_type_series):                                                                                                 
    rss = {}    
    for ct in cell_type_series.unique():                                                                                                  
        mask = (cell_type_series == ct).astype(float)
        p = mask / mask.sum()  # cell type 분포                                                                                           
        rss[ct] = {}                                                                                                                      
        for tf in auc_data.columns:                                                                                                       
            q = auc_data[tf] / auc_data[tf].sum()  # regulon activity 분포                                                                
            rss[ct][tf] = 1 - jensenshannon(p, q)                                                                                         
    return pd.DataFrame(rss)                                                                                                              

rss_df = calc_rss(auc_data, adata.obs['celltype'])                                                                                        
# Classical_monocyte 특이적 TF top 10
rss_df['Special_monocyte'].sort_values(ascending=False).head(10)  

In [ ]:
# UMAP이 adata에 있다고 가정
top_tfs = ['FOSL1(+)', 'BCL3(+)', 'TRIM28(+)', 'NFKB2(+)']                                                                                   

for tf in top_tfs:                                                                                                                        
    adata.obs[tf] = auc_data[tf].reindex(adata.obs.index)                                                                                 

sc.pl.umap(adata, color=top_tfs, ncols=2, cmap='Reds', vmin=0) 

In [ ]:
top_tfs = ['FOSL1(+)','NFKB2(+)','ATF5(+)','RELB(+)','NFKB1(+)']

for tf in top_tfs:
    adata.obs[tf] = auc_data[tf].reindex(adata.obs.index)

sc.pl.umap(
    adata,
    color=top_tfs,
    ncols=2,
    cmap='viridis',
    vmin=0,
    show=False
)

plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2a_2_v3.pdf'),
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [ ]:
sc.pl.umap(adata,color='FOSL1(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,) 
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_FOSL1.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='NFKB2(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)  
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_NFKB2.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='ATF5(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_ATF5.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='RELB(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_RELB.pdf'), bbox_inches='tight')
plt.close()

sc.pl.umap(adata,color='NFKB1(+)',ncols=2, cmap='viridis', vmin=0, show=False, return_fig=True,)   
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3A_NFKB1.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
# UMAP이 adata에 있다고 가정
top_tfs = ['FOSL1(+)','NFKB2(+)','ATF5(+)','RELB(+)','NFKB1(+)']                                                                                   

for tf in top_tfs:                                                                                                                        
    adata.obs[tf] = auc_data[tf].reindex(adata.obs.index)                                                                                 

sc.pl.umap(adata, color=top_tfs, ncols=2, cmap='viridis', vmin=0) 

In [ ]:
# ── FOSL1 / NFKB2 / ATF5 / RELB / NFKB1 regulon 추출 & SLC7A5 포함 여부 확인 ──
TARGET_TFS_5 = ['FOSL1', 'NFKB2', 'ATF5', 'RELB', 'NFKB1']

# regulons 딕셔너리에서 target genes 추출 (key: 'TF' 또는 'TF(+)' 모두 처리)
target_regulons_5 = {}
missing_tfs = []
for tf in TARGET_TFS_5:
    key_plus = tf + '(+)'
    if tf in regulons:
        target_regulons_5[tf] = list(set(regulons[tf]))
    elif key_plus in regulons:
        target_regulons_5[tf] = list(set(regulons[key_plus]))
    else:
        missing_tfs.append(tf)
        target_regulons_5[tf] = []

if missing_tfs:
    print(f'regulons에 없는 TF: {missing_tfs}')

# Summary
print('=== Regulon Summary (5 Target TFs) ===')
for tf, genes in target_regulons_5.items():
    slc_mark = '  ← SLC7A5 포함' if 'SLC7A5' in genes else ''
    print(f'  {tf:8s}: {len(genes):4d} target genes{slc_mark}')

# ── SLC7A5 포함 여부 확인 (5개 TF) ────────────────────────────────────────
print('\n=== SLC7A5가 5개 TF regulon에 포함된 TF ===')
slc7a5_tfs = [tf for tf, genes in target_regulons_5.items() if 'SLC7A5' in genes]

if slc7a5_tfs:
    for tf in slc7a5_tfs:
        print(f'  ✓ {tf} regulon → SLC7A5 포함')
else:
    print('  → 5개 TF 중 SLC7A5를 target으로 갖는 TF 없음')

# ── 전체 regulons에서 SLC7A5를 target으로 갖는 TF 전수 조사 ──────────────
print('\n=== 전체 pySCENIC regulons 중 SLC7A5를 target으로 갖는 TF 전수 조사 ===')
all_slc7a5_tfs = [tf for tf, genes in regulons.items() if 'SLC7A5' in genes]
if all_slc7a5_tfs:
    print(f'  발견된 TF ({len(all_slc7a5_tfs)}개): {all_slc7a5_tfs}')
else:
    print('  → 전체 regulons에서 SLC7A5를 target으로 갖는 TF 없음')

In [ ]:
# ── 5개 TF AUC: Special_monocyte vs Classical_monocyte 비교 violin ──────────
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

TARGET_TFS_5 = ['FOSL1', 'NFKB2', 'ATF5', 'RELB', 'NFKB1']
auc_cols_5   = [tf + '(+)' for tf in TARGET_TFS_5]

# Special_monocyte & Classical_monocyte 필터링
mask = adata.obs['celltype'].isin(['Special_monocyte', 'Classical_monocyte'])
auc_sub = auc_data.loc[mask, auc_cols_5].copy()
auc_sub['celltype'] = adata.obs.loc[mask, 'celltype']

# ── Mann-Whitney U test (Classical < Special 방향) ────────────────────────
pvals = []
for col in auc_cols_5:
    a = auc_sub.loc[auc_sub['celltype'] == 'Classical_monocyte', col].dropna()
    b = auc_sub.loc[auc_sub['celltype'] == 'Special_monocyte',   col].dropna()
    _, p = mannwhitneyu(a, b, alternative='less')  # Classical < Special
    pvals.append(p)

_, padj, _, _ = multipletests(pvals, method='fdr_bh')
pval_dict = {col: padj[i] for i, col in enumerate(auc_cols_5)}

def pval_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Long format ────────────────────────────────────────────────────────────
df_long = auc_sub.melt(id_vars='celltype', value_vars=auc_cols_5,
                        var_name='TF', value_name='AUC')
df_long['TF'] = df_long['TF'].str.replace(r'\(\+\)', '', regex=True)

# ── Violin plot ────────────────────────────────────────────────────────────
palette = {'Classical_monocyte': '#4393C3', 'Special_monocyte': '#D6604D'}
ct_order = ['Classical_monocyte', 'Special_monocyte']

fig, axes = plt.subplots(1, len(TARGET_TFS_5), figsize=(14, 4.5), sharey=False)

for ax, tf, col in zip(axes, TARGET_TFS_5, auc_cols_5):
    grp = df_long[df_long['TF'] == tf]
    sns.violinplot(data=grp, x='celltype', y='AUC',
                   palette=palette, inner='box',
                   order=ct_order, ax=ax, linewidth=0.8)
    ax.set_title(tf, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(['Classical', 'Special'], rotation=30,
                        ha='right', fontsize=8)

    # p-value annotation
    stars   = pval_stars(pval_dict[col])
    padj_v  = pval_dict[col]
    y_top   = grp['AUC'].max()
    y_range = grp['AUC'].max() - grp['AUC'].min()
    y_line  = y_top + y_range * 0.05

    x0, x1 = 0, 1
    ax.plot([x0, x0, x1, x1],
            [y_line, y_line + y_range*0.03, y_line + y_range*0.03, y_line],
            lw=0.8, color='black')
    ax.text(0.5, y_line + y_range * 0.05,
            f'{stars} (padj={padj_v:.2e})',
            ha='center', va='bottom', fontsize=7.5)

axes[0].set_ylabel('AUC score', fontsize=10)
for ax in axes[1:]:
    ax.set_ylabel('')

fig.suptitle(
    'Regulon Activity (AUC): Special_monocyte > Classical_monocyte\n'
    'FOSL1 / NFKB2 / ATF5 / RELB / NFKB1  |  Mann-Whitney U, FDR-BH',
    fontsize=11, y=1.04
)
plt.tight_layout()
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3_5TF_AUC_Special_vs_Classical.pdf'),
    bbox_inches='tight', dpi=300
)
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig3_5TF_AUC_Special_vs_Classical.png'),
    bbox_inches='tight', dpi=150
)
plt.show()

# ── 수치 요약 ───────────────────────────────────────────────────────────────
print("=== Mean AUC 비교 ===")
for col in auc_cols_5:
    tf = col.replace('(+)', '')
    m_cls = auc_sub.loc[auc_sub['celltype']=='Classical_monocyte', col].mean()
    m_spc = auc_sub.loc[auc_sub['celltype']=='Special_monocyte',   col].mean()
    stars = pval_stars(pval_dict[col])
    print(f"  {tf:8s}  Classical={m_cls:.4f}  Special={m_spc:.4f}  "
          f"FC={m_spc/m_cls:.2f}x  {stars}")

# Classical monocyte vs SLC7A5 high monocyte comparison

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ── 그룹 정의 ───────────────────────────────────────────────
cluster_A_cells = adata.obs[adata.obs['celltype'] == 'Classical_monocyte'].index
cluster_B_cells = adata.obs[adata.obs['celltype'] == 'Special_monocyte'].index

print(f"Classical_monocyte  : {len(cluster_A_cells)} cells")
print(f"SLC7A5_high_monocyte: {len(cluster_B_cells)} cells")

# ── Mann-Whitney U test on AUC scores ─────────────────────────
auc = auc_matrix.copy()   # cell 7에서 로드한 auc_matrix 재사용

results_slc = []
for tf in auc.columns:
    a = auc.loc[auc.index.isin(cluster_A_cells), tf].dropna()
    b = auc.loc[auc.index.isin(cluster_B_cells), tf].dropna()
    if len(a) < 3 or len(b) < 3:
        continue
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    fc = (b.mean() + 1e-10) / (a.mean() + 1e-10)   # SLC7A5_high / Classical
    results_slc.append({
        'TF'              : tf,
        'mean_Classical'  : a.mean(),
        'mean_SLC7A5_high': b.mean(),
        'log2FC'          : np.log2(fc),   # + = SLC7A5_high 높음
        'pval'            : p
    })

df = pd.DataFrame(results_slc).set_index('TF')
df['padj'] = multipletests(df['pval'], method='fdr_bh')[1]
df['sig']  = (df['padj'] < 0.05) & (df['log2FC'].abs() > 0.5)
df_sig     = df[df['padj'] < 0.05].copy()

print(f"Significant TFs (padj < 0.05): {len(df_sig)}")
df.sort_values('log2FC', ascending=False).head(10)

In [ ]:
# Classical vs SLC7A5_high 두 그룹 RSS 계산 (cell 26의 calc_rss 재사용)
auc_sub = auc.loc[list(cluster_A_cells) + list(cluster_B_cells)]

group_labels = pd.concat([
    pd.Series('Classical',   index=cluster_A_cells),
    pd.Series('SLC7A5_high', index=cluster_B_cells)
])

rss_slc = calc_rss(auc_sub, group_labels)   # index: TF, columns: Classical / SLC7A5_high
rss_slc

In [ ]:
# RSS 상위 TF 확인
print("=== Classical 특이적 TF (RSS top 10) ===")
print(rss_slc['Classical'].nlargest(10).to_string())
print()
print("=== SLC7A5_high 특이적 TF (RSS top 10) ===")
print(rss_slc['SLC7A5_high'].nlargest(10).to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

not_sig = df[~df['sig']]
sig_A   = df[df['sig'] & (df['log2FC'] > 0)]   # SLC7A5_high 높음
sig_B   = df[df['sig'] & (df['log2FC'] < 0)]   # Classical 높음

ax.scatter(not_sig['log2FC'], -np.log10(not_sig['padj']),
           color='grey', alpha=0.4, s=20, label='Not significant')
ax.scatter(sig_A['log2FC'], -np.log10(sig_A['padj']),
           color='#E74C3C', alpha=0.7, s=30, label='SLC7A5_high enriched')
ax.scatter(sig_B['log2FC'], -np.log10(sig_B['padj']),
           color='#3498DB', alpha=0.7, s=30, label='Classical enriched')

for _, row in sig_A.nlargest(5, 'log2FC').iterrows():
    ax.annotate(row.name.replace('(+)', ''),
                (row['log2FC'], -np.log10(row['padj'])),
                xytext=(5, 2), textcoords='offset points', fontsize=8, color='#922B21')
for _, row in sig_B.nsmallest(5, 'log2FC').iterrows():
    ax.annotate(row.name.replace('(+)', ''),
                (row['log2FC'], -np.log10(row['padj'])),
                xytext=(5, 2), textcoords='offset points', fontsize=8, color='#1A5276')

ax.axhline(-np.log10(0.05), color='black', linestyle='--', lw=0.8, alpha=0.5)
ax.axvline(-0.5, color='grey', linestyle='--', lw=0.8, alpha=0.5)
ax.axvline( 0.5, color='grey', linestyle='--', lw=0.8, alpha=0.5)
ax.set_xlabel('log2FC (SLC7A5_high / Classical)', fontsize=11)
ax.set_ylabel('-log10(adjusted p-value)', fontsize=11)
ax.set_title('TF Regulon Activity\nClassical vs SLC7A5_high monocyte', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# (volcano는 위 셀에서 plt.show()로 출력됨)

In [ ]:
top_tfs_clust = (df_sig
                 .reindex(df_sig['log2FC'].abs().sort_values(ascending=False).index)
                 .head(30).index.tolist())

plot_data = auc.loc[list(cluster_A_cells) + list(cluster_B_cells), top_tfs_clust].copy()
plot_data.columns = plot_data.columns.str.replace('(+)', '', regex=False)

row_colors = (['#3498DB'] * len(cluster_A_cells) +   # Classical  = blue
              ['#E74C3C'] * len(cluster_B_cells))     # SLC7A5_high = red

# z_score 정규화 시 std=0 행 제거
plot_data_T = plot_data.T
plot_data_T = plot_data_T[plot_data_T.std(axis=1) > 0]

g = sns.clustermap(plot_data_T, col_colors=row_colors,
                   cmap='RdBu_r', z_score=0,
                   xticklabels=False, yticklabels=True,
                   figsize=(12, 8))
g.ax_heatmap.set_ylabel('TF (Regulon)', fontsize=10)
plt.show()

In [ ]:
top_A_tfs = rss_slc['Classical'].nlargest(15).index.tolist()
top_B_tfs = rss_slc['SLC7A5_high'].nlargest(15).index.tolist()
top_tfs_rss = list(dict.fromkeys(top_A_tfs + top_B_tfs))   # 중복 제거, 순서 유지

rss_plot = rss_slc.loc[top_tfs_rss].copy()
rss_plot.index = rss_plot.index.str.replace('(+)', '', regex=False)

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(4, len(top_tfs_rss) * 0.4 + 1), dpi=150)
fig.patch.set_facecolor('white')
sns.heatmap(rss_plot, cmap='YlOrRd', ax=ax,
            linewidths=0.3, linecolor='lightgrey',
            cbar_kws={'label': 'RSS', 'shrink': 0.5},
            annot=True, fmt='.3f')
ax.set_title('RSS: Classical vs SLC7A5_high', fontsize=11)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

adj = pd.read_csv(
    artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv'),
    sep='\t'
)
# adjacencies TSV 컬럼 확인
print(adj.columns.tolist())

sig_tfs_clean      = set(df_sig[df_sig['log2FC'] >  0].index.str.replace('(+)', '', regex=False))  # SLC7A5_high
classical_tfs_clean = set(df_sig[df_sig['log2FC'] < 0].index.str.replace('(+)', '', regex=False))  # Classical

adj_sig = adj[adj['TF'].isin(sig_tfs_clean | classical_tfs_clean)].copy()

G = nx.from_pandas_edgelist(adj_sig, source='TF', target='target', edge_attr='importance')

node_colors = []
for node in G.nodes():
    if node in sig_tfs_clean:
        node_colors.append('#E74C3C')        # SLC7A5_high 특이적
    elif node in classical_tfs_clean:
        node_colors.append('#3498DB')        # Classical 특이적
    else:
        node_colors.append('#BDC3C7')        # target gene

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(12, 10), dpi=150)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

pos = nx.spring_layout(G, k=0.5, seed=42)
nx.draw_networkx(G, pos, ax=ax,
                 node_color=node_colors, node_size=300,
                 font_size=6, edge_color='#BDC3C7',
                 alpha=0.8, arrows=True, arrowsize=10)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#E74C3C', label='SLC7A5_high enriched TF'),
    Patch(facecolor='#3498DB', label='Classical enriched TF'),
    Patch(facecolor='#BDC3C7', label='Target gene'),
], loc='upper left', fontsize=9)
ax.set_title('TF Regulon Network\n(Classical vs SLC7A5_high monocyte)', fontsize=12)
plt.tight_layout()
plt.show()

# Figure — Classical vs SLC7A5_high monocyte (Publication-ready)
> Volcano plot / RSS dot plot / Clustermap / Mirror barplot

In [ ]:
# ── Figure: Volcano plot (Classical vs SLC7A5_high, TF Regulon Activity) ──
# adjustText 없으면: pip install adjustText
from adjustText import adjust_text
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10})

not_sig = df[~df['sig']]
sig_A   = df[df['sig'] & (df['log2FC'] > 0)]   # SLC7A5_high 높음
sig_B   = df[df['sig'] & (df['log2FC'] < 0)]   # Classical 높음

fig, ax = plt.subplots(figsize=(7, 6), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.spines[['top', 'right']].set_visible(False)

ax.scatter(not_sig['log2FC'], -np.log10(not_sig['padj']),
           color='#BDBDBD', alpha=0.5, s=18, linewidths=0, label='Not significant', zorder=2)
ax.scatter(sig_A['log2FC'], -np.log10(sig_A['padj']),
           color='#C0392B', alpha=0.85, s=40, linewidths=0, label='SLC7A5_high enriched', zorder=3)
ax.scatter(sig_B['log2FC'], -np.log10(sig_B['padj']),
           color='#2471A3', alpha=0.85, s=40, linewidths=0, label='Classical enriched', zorder=3)

texts = []
for _, row in pd.concat([sig_A.nlargest(6, 'log2FC'), sig_B.nsmallest(6, 'log2FC')]).iterrows():
    col = '#922B21' if row['log2FC'] > 0 else '#1A5276'
    texts.append(ax.text(row['log2FC'], -np.log10(row['padj']),
                         row.name.replace('(+)', ''),
                         fontsize=8.5, color=col, fontweight='bold'))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='grey', lw=0.6))

ax.axhline(-np.log10(0.05), color='#555', linestyle='--', lw=0.8, alpha=0.6, zorder=1)
ax.axvline(-0.5, color='#AAAAAA', linestyle='--', lw=0.8, zorder=1)
ax.axvline( 0.5, color='#AAAAAA', linestyle='--', lw=0.8, zorder=1)
ax.set_xlabel(r'log$_2$FC (SLC7A5_high / Classical)', fontsize=12)
ax.set_ylabel(r'$-$log$_{10}$(adjusted p-value)', fontsize=12)
ax.set_title('TF Regulon Activity\nClassical vs SLC7A5_high monocyte', fontsize=12, pad=10)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
#plt.savefig('Figure_volcano_TF.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure: RSS Dot plot (Classical vs SLC7A5_high) ──
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10})

top_A_tfs = rss_slc['Classical'].nlargest(15).index.tolist()
top_B_tfs = rss_slc['SLC7A5_high'].nlargest(15).index.tolist()
top_tfs_rss = list(dict.fromkeys(top_A_tfs + top_B_tfs))

rss_plot = rss_slc.loc[top_tfs_rss].copy()
rss_plot.index = rss_plot.index.str.replace('(+)', '', regex=False)

fig, ax = plt.subplots(figsize=(3.5, len(top_tfs_rss) * 0.42 + 1.5), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)

col_colors = {'Classical': '#2471A3', 'SLC7A5_high': '#C0392B'}
for xi, col in enumerate(rss_plot.columns):
    vals = rss_plot[col]
    max_val = rss_plot.values.max()
    sizes = (vals / max_val) * 450
    ax.scatter([xi] * len(vals), range(len(vals)),
               s=sizes, color=col_colors[col], alpha=0.85, linewidths=0)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Classical', 'SLC7A5_high'], fontsize=11, fontweight='bold')
ax.set_yticks(range(len(rss_plot)))
ax.set_yticklabels(rss_plot.index, fontsize=9)
ax.set_xlim(-0.5, 1.5)
ax.invert_yaxis()
ax.set_title('RSS: Classical vs SLC7A5_high', fontsize=11, pad=10)
ax.grid(axis='x', alpha=0.12, linestyle='--')
ax.tick_params(length=0)

# 범례 (점 크기 가이드)
for size_label, rss_val in zip(['Low', 'Mid', 'High'], [0.2, 0.5, 0.9]):
    ax.scatter([], [], s=(rss_val / 1.0) * 450, color='grey', alpha=0.7,
               label=f'RSS={rss_val}', linewidths=0)
ax.legend(title='RSS scale', frameon=False, fontsize=8, title_fontsize=8,
          loc='lower right', bbox_to_anchor=(1.4, 0))

plt.tight_layout()
plt.savefig('Figure_RSS_dotplot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure: Clustermap (Top 30 TF AUC, Classical vs SLC7A5_high) ──
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10})
sns.reset_defaults()

top_tfs_clust = (df_sig
                 .reindex(df_sig['log2FC'].abs().sort_values(ascending=False).index)
                 .head(30).index.tolist())

plot_data = auc.loc[list(cluster_A_cells) + list(cluster_B_cells), top_tfs_clust].copy()
plot_data.columns = plot_data.columns.str.replace('(+)', '', regex=False)

col_lut = {'Classical': '#2471A3', 'SLC7A5_high': '#C0392B'}
col_colors_series = pd.Series(
    ['Classical'] * len(cluster_A_cells) + ['SLC7A5_high'] * len(cluster_B_cells),
    index=list(cluster_A_cells) + list(cluster_B_cells)
).map(col_lut)

plot_data_T = plot_data.T
plot_data_T = plot_data_T[plot_data_T.std(axis=1) > 0]

g = sns.clustermap(plot_data_T,
                   col_colors=col_colors_series,
                   cmap='RdBu_r', z_score=0,
                   xticklabels=False, yticklabels=True,
                   figsize=(12, 8), dendrogram_ratio=(0.08, 0.12),
                   cbar_pos=(1.02, 0.3, 0.02, 0.3),
                   linewidths=0)
g.ax_heatmap.set_ylabel('TF Regulon', fontsize=11)
g.ax_heatmap.set_xlabel('')
g.ax_cbar.set_title('z-score', fontsize=9, pad=6)

legend_els = [Patch(facecolor=v, label=k) for k, v in col_lut.items()]
g.ax_col_dendrogram.legend(handles=legend_els, loc='center left',
                            bbox_to_anchor=(1.04, 0.5), frameon=False, fontsize=10)
g.fig.suptitle('TF Regulon AUC — Classical vs SLC7A5_high', fontsize=12, y=1.01)
plt.savefig('Figure_clustermap_TF.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure: Mirror barplot (Top 10 TF by RSS, each group) ──
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10})

top_A = rss_slc['Classical'].nlargest(10)
top_B = rss_slc['SLC7A5_high'].nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(9, 5), dpi=200)
fig.patch.set_facecolor('white')

for ax, data, color, title in zip(
        axes,
        [top_A, top_B],
        ['#2471A3', '#C0392B'],
        ['Classical monocyte', 'SLC7A5_high monocyte']):

    labels = data.index.str.replace('(+)', '', regex=False)[::-1]
    values = data.values[::-1]

    bars = ax.barh(range(len(labels)), values, color=color, alpha=0.85, height=0.6)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('RSS', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', color=color, pad=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='y', length=0)

    # 값 라벨
    for bar, val in zip(bars, values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=7.5, color='#333')

fig.suptitle('Top TF Regulons by RSS', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('Figure_mirror_barplot_RSS.pdf', bbox_inches='tight')
plt.show()

# Integrated Figure — Classical vs Special monocyte TF & Signaling
> **Panel A** RSS Scatter (specificity landscape) | **Panel B** Integrated Score Bar (top TFs) | **Panel C** KEGG Pathway Enrichment (downstream signaling)
>
> 실행 순서: Cell 52 → 53 → 12 완료 후 아래 두 셀 순서대로 실행

In [ ]:
# ── Data prep: Integrated score + KEGG enrichment ──────────────
import pandas as pd
import numpy as np
import gseapy as gp
import warnings
warnings.filterwarnings('ignore')

# 1. Integrated score = RSS_diff × -log10(padj)
#    양수: Special 특이적 / 음수: Classical 특이적
df_int = df.copy()
df_int['RSS_Classical'] = rss_slc['Classical']
df_int['RSS_SLC7A5']    = rss_slc['SLC7A5_high']
df_int['RSS_diff']      = rss_slc['SLC7A5_high'] - rss_slc['Classical']
df_int['-log10padj']    = -np.log10(df_int['padj'].clip(lower=1e-300))
df_int['integrated_score'] = df_int['RSS_diff'] * df_int['-log10padj']

# 2. Significant TF → target genes (regulons dict, Cell 12)
classical_tfs = df_sig[df_sig['log2FC'] < -0.5].index.str.replace('(+)', '', regex=False).tolist()
special_tfs   = df_sig[df_sig['log2FC'] >  0.5].index.str.replace('(+)', '', regex=False).tolist()

classical_targets = list(set(sum([regulons.get(tf, []) for tf in classical_tfs], [])))
special_targets   = list(set(sum([regulons.get(tf, []) for tf in special_tfs],   [])))

print(f"Classical TFs: {len(classical_tfs):>3}  →  target genes: {len(classical_targets)}")
print(f"Special   TFs: {len(special_tfs):>3}  →  target genes: {len(special_targets)}")

# 3. KEGG Pathway Enrichment
def run_enrichr(gene_list, label):
    if len(gene_list) == 0:
        print(f"[{label}] No genes — skipping enrichment")
        return pd.DataFrame(columns=['Term', 'Adjusted P-value', '-log10padj'])
    try:
        enr = gp.enrichr(gene_list=gene_list, gene_sets=['KEGG_2021_Human'],
                         organism='human', outdir=None, verbose=False)
        res = enr.results[enr.results['Adjusted P-value'] < 0.05].head(10).copy()
        res['-log10padj'] = -np.log10(res['Adjusted P-value'].clip(lower=1e-300))
        print(f"[{label}] Significant pathways: {len(res)}")
        return res
    except Exception as e:
        print(f"[{label}] Enrichment error: {e}")
        return pd.DataFrame(columns=['Term', 'Adjusted P-value', '-log10padj'])

enr_classical_df = run_enrichr(classical_targets, 'Classical')
enr_special_df   = run_enrichr(special_targets,   'Special')

In [ ]:
# ── Integrated Figure: TF Landscape + Signaling (3-panel) ───────
from adjustText import adjust_text
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8, 'xtick.major.width': 0.8,
                     'ytick.major.width': 0.8})

CLR = '#2471A3'   # Classical
SLC = '#C0392B'   # Special / SLC7A5_high

fig = plt.figure(figsize=(18, 7), dpi=200)
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.42, left=0.06, right=0.97, top=0.92, bottom=0.12)
ax_A = fig.add_subplot(gs[0])
ax_B = fig.add_subplot(gs[1])
ax_C = fig.add_subplot(gs[2])

# ═══════════════════════════════════════════════════════════
# Panel A — RSS Scatter: TF Specificity Landscape
# ═══════════════════════════════════════════════════════════
for ax in [ax_A, ax_B, ax_C]:
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible('False')

sig_sp = df_int[df_int['sig'] & (df_int['log2FC'] > 0)]
sig_cl = df_int[df_int['sig'] & (df_int['log2FC'] < 0)]
ns     = df_int[~df_int['sig']]

ax_A.scatter(ns['RSS_Classical'], ns['RSS_SLC7A5'],
             c='#BDBDBD', s=12, alpha=0.4, linewidths=0, zorder=2)
ax_A.scatter(sig_cl['RSS_Classical'], sig_cl['RSS_SLC7A5'],
             c=CLR, s=sig_cl['-log10padj'] * 7, alpha=0.85, linewidths=0,
             label='Classical enriched', zorder=3)
ax_A.scatter(sig_sp['RSS_Classical'], sig_sp['RSS_SLC7A5'],
             c=SLC, s=sig_sp['-log10padj'] * 7, alpha=0.85, linewidths=0,
             label='Special enriched', zorder=3)

lim = df_int[['RSS_Classical', 'RSS_SLC7A5']].values.max() * 1.08
ax_A.plot([0, lim], [0, lim], '--', color='#888', lw=0.9, alpha=0.6, zorder=1)
ax_A.set_xlim(-0.002, lim); ax_A.set_ylim(-0.002, lim)

label_tfs = pd.concat([
    sig_sp.nlargest(5, 'integrated_score'),
    sig_cl.nsmallest(5, 'integrated_score')
])
texts = []
for _, row in label_tfs.iterrows():
    col = SLC if row['log2FC'] > 0 else CLR
    texts.append(ax_A.text(row['RSS_Classical'], row['RSS_SLC7A5'],
                           row.name.replace('(+)', ''),
                           fontsize=8, color=col, fontweight='bold'))
adjust_text(texts, ax=ax_A, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

ax_A.set_xlabel('RSS — Classical monocyte', fontsize=11)
ax_A.set_ylabel('RSS — Special monocyte', fontsize=11)
ax_A.set_title('A   TF Specificity Landscape', fontsize=11, fontweight='bold', loc='left', pad=8)
ax_A.legend(frameon=False, fontsize=8.5, loc='upper left',
            markerscale=0.7, handletextpad=0.3)

# ═══════════════════════════════════════════════════════════
# Panel B — Integrated Score Bar: Top Differentially Active TFs
# ═══════════════════════════════════════════════════════════
top_sp_bar = df_int[df_int['sig'] & (df_int['log2FC'] > 0)].nlargest(10, 'integrated_score')
top_cl_bar = df_int[df_int['sig'] & (df_int['log2FC'] < 0)].nsmallest(10, 'integrated_score')
combined   = pd.concat([top_cl_bar, top_sp_bar]).sort_values('integrated_score')
labels     = combined.index.str.replace('(+)', '', regex=False)
colors     = [CLR if s < 0 else SLC for s in combined['integrated_score']]

bars = ax_B.barh(range(len(combined)), combined['integrated_score'],
                 color=colors, alpha=0.85, height=0.65, linewidth=0)
ax_B.set_yticks(range(len(combined)))
ax_B.set_yticklabels(labels, fontsize=8.5)
ax_B.axvline(0, color='black', lw=0.8)
ax_B.tick_params(axis='y', length=0)
ax_B.set_xlabel('Integrated Score  (RSSΔ × −log₁₀ pₐₑⱼ)', fontsize=10)
ax_B.set_title('B   Top Differentially Active TFs', fontsize=11, fontweight='bold', loc='left', pad=8)

val_range = combined['integrated_score'].abs().max()
for bar, val in zip(bars, combined['integrated_score']):
    offset = val_range * 0.025
    ax_B.text(val + offset if val >= 0 else val - offset,
              bar.get_y() + bar.get_height() / 2,
              f'{val:.2f}', va='center', fontsize=6.5,
              ha='left' if val >= 0 else 'right', color='#444')

# ═══════════════════════════════════════════════════════════
# Panel C — KEGG Pathway Lollipop: Downstream Signaling
# ═══════════════════════════════════════════════════════════
def shorten(name, n=38):
    s = name.split(' KEGG')[0].split(' Homo')[0]
    return s[:n] + '..' if len(s) > n else s

rows = []
for _, r in enr_classical_df.iterrows():
    rows.append({'Term': shorten(r['Term']), 'score': -r['-log10padj'], 'group': 'Classical'})
for _, r in enr_special_df.iterrows():
    rows.append({'Term': shorten(r['Term']), 'score':  r['-log10padj'], 'group': 'Special'})

if rows:
    all_paths = pd.DataFrame(rows).sort_values('score')
    lcolors   = [CLR if g == 'Classical' else SLC for g in all_paths['group']]
    ypos = range(len(all_paths))
    ax_C.hlines(y=ypos, xmin=0, xmax=all_paths['score'].values,
                colors=lcolors, linewidth=1.4, alpha=0.75)
    ax_C.scatter(all_paths['score'], ypos, c=lcolors, s=55, zorder=3, linewidths=0)
    ax_C.set_yticks(ypos)
    ax_C.set_yticklabels(all_paths['Term'].values, fontsize=7.5)
    ax_C.axvline(0, color='black', lw=0.8)
    ax_C.tick_params(axis='y', length=0)
    max_v = all_paths['score'].abs().max() * 1.15
    ax_C.set_xlim(-max_v, max_v)
    ax_C.set_xlabel('−log₁₀(adjusted p-value)\n← Classical       Special →', fontsize=10)
else:
    ax_C.text(0.5, 0.5, 'No significant pathways\n(padj < 0.05)',
              ha='center', va='center', transform=ax_C.transAxes, fontsize=10, color='grey')
    ax_C.set_xticks([])
    ax_C.set_yticks([])

ax_C.set_title('C   Downstream Signaling (KEGG)', fontsize=11, fontweight='bold', loc='left', pad=8)

# ── size legend for Panel A ────────────────────────────────
from matplotlib.lines import Line2D
size_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='grey',
           markersize=np.sqrt(v * 7), label=f'−log₁₀p={v:.0f}', alpha=0.7)
    for v in [2, 5, 10]
]
ax_A.legend(handles=size_legend + ax_A.get_legend_handles_labels()[0],
            labels=[f'−log₁⁰p={v:.0f}' for v in [2, 5, 10]] + ax_A.get_legend_handles_labels()[1],
            frameon=False, fontsize=8, loc='upper left', title='dot size',
            title_fontsize=7.5, handletextpad=0.2, labelspacing=0.3)

fig.suptitle('Classical vs Special monocyte — TF Regulon & Downstream Signaling',
             fontsize=13, fontweight='bold')
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Figure_integrated_TF_signaling.pdf'), bbox_inches='tight')
plt.show()
print("Saved: Figure_integrated_TF_signaling.pdf")

# Dual-evidence Figure — Classical vs Special monocyte (Z-score ∩ RSS)
> **Panel A** Dual-evidence scatter (log₂FC vs ΔRSS)  
> **Panel B** Concordance-score barplot (top TFs agreed by both methods)  
> **Panel C** AUC dot plot (activity across all cell types for selected TFs)  
>
> 전제: cell 52 (Mann-Whitney, `df`/`df_sig`), cell 53 (RSS, `rss_slc`) 완료 후 실행

In [ ]:
# ── Dual-evidence: build combined dataframe ──────────────────
import pandas as pd
import numpy as np

df_comb = df.copy()
df_comb['RSS_Classical'] = rss_slc['Classical']
df_comb['RSS_Special']   = rss_slc['SLC7A5_high']
df_comb['delta_RSS']     = rss_slc['SLC7A5_high'] - rss_slc['Classical']
df_comb['-log10padj']    = -np.log10(df_comb['padj'].clip(lower=1e-300))

# Concordant: log2FC and delta_RSS point in the same direction AND sig
df_comb['concordant'] = (
    ((df_comb['log2FC'] > 0) & (df_comb['delta_RSS'] > 0)) |
    ((df_comb['log2FC'] < 0) & (df_comb['delta_RSS'] < 0))
) & df_comb['sig']

df_comb['group'] = 'Discordant/NS'
df_comb.loc[df_comb['concordant'] & (df_comb['log2FC'] > 0), 'group'] = 'Special (concordant)'
df_comb.loc[df_comb['concordant'] & (df_comb['log2FC'] < 0), 'group'] = 'Classical (concordant)'

# Triple concordance score: log2FC × delta_RSS × -log10padj
df_comb['triple_score'] = df_comb['log2FC'] * df_comb['delta_RSS'] * df_comb['-log10padj']

# Percentile ranks (0=Classical direction, 100=Special direction)
df_comb['rank_fc']  = df_comb['log2FC'].rank(pct=True) * 100
df_comb['rank_rss'] = df_comb['delta_RSS'].rank(pct=True) * 100

sp_con = df_comb[df_comb['group'] == 'Special (concordant)']
cl_con = df_comb[df_comb['group'] == 'Classical (concordant)']

print(f"Concordant Special TFs  : {len(sp_con)}")
print(f"Concordant Classical TFs: {len(cl_con)}")
print()
print("=== Top Special (concordant) ===")
print(sp_con.nlargest(10, 'triple_score')[['log2FC','delta_RSS','-log10padj','triple_score']].to_string())
print()
print("=== Top Classical (concordant) ===")
print(cl_con.nsmallest(10, 'triple_score')[['log2FC','delta_RSS','-log10padj','triple_score']].to_string())


In [ ]:
# ── Dual-evidence Figure: 3-panel ────────────────────────────
from adjustText import adjust_text
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8, 'xtick.major.width': 0.8,
                     'ytick.major.width': 0.8})

CLR  = '#2471A3'   # Classical = blue
SLC  = '#C0392B'   # Special   = red
DISC = '#BDBDBD'   # Discordant / NS

fig = plt.figure(figsize=(18, 6.5), dpi=200)
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(1, 3, figure=fig,
                       wspace=0.42, left=0.06, right=0.97, top=0.90, bottom=0.12)
ax_A = fig.add_subplot(gs[0])
ax_B = fig.add_subplot(gs[1])
ax_C = fig.add_subplot(gs[2])

for ax in [ax_A, ax_B, ax_C]:
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

# ═══════════════════════════════════════════════════════════════
# Panel A — Dual-evidence scatter: log2FC (x) vs delta-RSS (y)
# Each dot = 1 TF; size ∝ -log10(padj)
# ═══════════════════════════════════════════════════════════════
ns     = df_comb[df_comb['group'] == 'Discordant/NS']

ax_A.scatter(ns['log2FC'], ns['delta_RSS'],
             c=DISC, s=14, alpha=0.35, linewidths=0, zorder=2)
ax_A.scatter(cl_con['log2FC'], cl_con['delta_RSS'],
             c=CLR, s=cl_con['-log10padj'] * 9, alpha=0.85, linewidths=0,
             label='Classical (concordant)', zorder=3)
ax_A.scatter(sp_con['log2FC'], sp_con['delta_RSS'],
             c=SLC, s=sp_con['-log10padj'] * 9, alpha=0.85, linewidths=0,
             label='Special (concordant)', zorder=3)

ax_A.axhline(0, color='#888', lw=0.8, ls='--', alpha=0.5, zorder=1)
ax_A.axvline(0, color='#888', lw=0.8, ls='--', alpha=0.5, zorder=1)

# Quadrant labels
xlim = ax_A.get_xlim(); ylim = ax_A.get_ylim()
for ax_ref in [ax_A]:
    xmin, xmax = df_comb['log2FC'].min(), df_comb['log2FC'].max()
    ymin, ymax = df_comb['delta_RSS'].min(), df_comb['delta_RSS'].max()
    ax_ref.text(xmax * 0.85, ymax * 0.92, 'Special\n(concordant)',
                fontsize=7.5, color=SLC, ha='center', style='italic')
    ax_ref.text(xmin * 0.85, ymin * 0.92, 'Classical\n(concordant)',
                fontsize=7.5, color=CLR, ha='center', style='italic')

# Label top concordant TFs
label_tfs = pd.concat([
    sp_con.nlargest(6, 'triple_score'),
    cl_con.nsmallest(6, 'triple_score')
])
texts = []
for _, row in label_tfs.iterrows():
    col = SLC if row['log2FC'] > 0 else CLR
    texts.append(ax_A.text(row['log2FC'], row['delta_RSS'],
                            row.name.replace('(+)', ''),
                            fontsize=8.5, color=col, fontweight='bold'))
adjust_text(texts, ax=ax_A, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

ax_A.set_xlabel('log₂FC  (Special / Classical)', fontsize=11)
ax_A.set_ylabel('ΔRSS  (RSS_Special − RSS_Classical)', fontsize=11)
ax_A.set_title('A   Fold-change vs Specificity (RSS)', fontsize=11,
               fontweight='bold', loc='left', pad=8)
ax_A.legend(frameon=False, fontsize=8.5, loc='upper left', handletextpad=0.3)

# ═══════════════════════════════════════════════════════════════
# Panel B — Concordance-score barplot
# Triple score = log2FC × delta_RSS × -log10padj
# (both methods must agree, and stat sig boosts it)
# ═══════════════════════════════════════════════════════════════
top_sp_bar = sp_con.nlargest(10, 'triple_score')
top_cl_bar = cl_con.nsmallest(10, 'triple_score')
bar_data   = pd.concat([top_cl_bar, top_sp_bar]).sort_values('triple_score')
bar_labels = bar_data.index.str.replace('(+)', '', regex=False)
bar_colors = [CLR if s < 0 else SLC for s in bar_data['triple_score']]

bars = ax_B.barh(range(len(bar_data)), bar_data['triple_score'],
                 color=bar_colors, alpha=0.85, height=0.65, linewidth=0)
ax_B.set_yticks(range(len(bar_data)))
ax_B.set_yticklabels(bar_labels, fontsize=8.5)
ax_B.axvline(0, color='black', lw=0.8)
ax_B.tick_params(axis='y', length=0)
val_range = bar_data['triple_score'].abs().max()
for bar, val in zip(bars, bar_data['triple_score']):
    offset = val_range * 0.03
    ax_B.text(val + offset if val >= 0 else val - offset,
              bar.get_y() + bar.get_height() / 2,
              f'{val:.3f}', va='center', fontsize=6.5,
              ha='left' if val >= 0 else 'right', color='#444')
ax_B.set_xlabel('Concordance Score  (log₂FC × ΔRSS × −log₁₀p)', fontsize=9.5)
ax_B.set_title('B   Top Concordant TFs (FC ∩ RSS)', fontsize=11,
               fontweight='bold', loc='left', pad=8)

# ═══════════════════════════════════════════════════════════════
# Panel C — Rank concordance dot plot
# Shows each TF as two dots: FC-percentile (○) & RSS-percentile (◆)
# Connected by line; gap = disagreement between the two metrics
# ═══════════════════════════════════════════════════════════════
top_sp_dot = sp_con.nlargest(8, 'triple_score')
top_cl_dot = cl_con.nsmallest(8, 'triple_score')
dot_data   = pd.concat([top_cl_dot, top_sp_dot]).sort_values('triple_score')
dot_labels = dot_data.index.str.replace('(+)', '', regex=False)
dot_colors = [CLR if s < 0 else SLC for s in dot_data['triple_score']]

ypos = np.arange(len(dot_data))
for i, (r_fc, r_rss, col) in enumerate(
        zip(dot_data['rank_fc'], dot_data['rank_rss'], dot_colors)):
    ax_C.plot([r_fc, r_rss], [ypos[i] - 0.15, ypos[i] + 0.15],
              color=col, alpha=0.45, lw=1.2)

ax_C.scatter(dot_data['rank_fc'],  ypos - 0.15,
             c=dot_colors, s=55, marker='o', alpha=0.95,
             linewidths=0, label='FC percentile  ●', zorder=4)
ax_C.scatter(dot_data['rank_rss'], ypos + 0.15,
             c=dot_colors, s=55, marker='D', alpha=0.95,
             linewidths=0, label='RSS percentile ◆', zorder=4)

ax_C.set_yticks(ypos)
ax_C.set_yticklabels(dot_labels, fontsize=8.5)
ax_C.axvline(50, color='#aaa', lw=0.7, ls='--', alpha=0.5)
ax_C.set_xlim(0, 108)
ax_C.set_xlabel('Percentile rank  (0 = Classical direction → 100 = Special)', fontsize=9.5)
ax_C.set_title('C   FC vs RSS Rank Concordance', fontsize=11,
               fontweight='bold', loc='left', pad=8)

from matplotlib.lines import Line2D
ax_C.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', markersize=7, label='FC percentile  ●'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor='grey', markersize=6, label='RSS percentile ◆'),
], frameon=False, fontsize=8.5, loc='lower right', handletextpad=0.2)

fig.suptitle('Classical vs Special monocyte — Dual-evidence TF Analysis (Z-score & RSS)',
             fontsize=13, fontweight='bold')

save_path = output_path('02_SLC7A5_mono_Journal/05_Figure/Figure_dual_evidence_TF.pdf')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")


In [ ]:
# ── Panel D (optional standalone): AUC dot plot across cell types
# Validates that concordant TFs are truly specific to expected cell type
# ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10, 'axes.linewidth': 0.8})

# Select top 6 concordant TFs per group
top_sp_tfs = sp_con.nlargest(6, 'triple_score').index.tolist()
top_cl_tfs = cl_con.nsmallest(6, 'triple_score').index.tolist()
sel_tfs    = top_cl_tfs + top_sp_tfs   # Classical on top, Special on bottom

# Mean AUC per celltype (all cells, not just the two monocytes)
auc_all   = auc_data   # from cell 13: auc_data = adata.obs[cols ending with '(+)']
celltypes = adata.obs['celltype']

# Build mean_auc and pct_expressing per (celltype, TF)
rows_dot = []
for ct in sorted(celltypes.unique()):
    mask = celltypes == ct
    for tf in sel_tfs:
        vals = auc_all.loc[mask, tf].dropna()
        if len(vals) == 0:
            continue
        rows_dot.append({
            'celltype': ct,
            'TF'      : tf.replace('(+)', ''),
            'mean_auc': vals.mean(),
            'pct_expr': (vals > vals.quantile(0.25)).mean() * 100   # % cells above Q1
        })

dot_df = pd.DataFrame(rows_dot)

# Z-score normalise mean_auc per TF (across cell types) for colour scale
dot_df['z_auc'] = dot_df.groupby('TF')['mean_auc'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-10))

# Pivot for plotting
piv_z   = dot_df.pivot(index='TF', columns='celltype', values='z_auc')
piv_pct = dot_df.pivot(index='TF', columns='celltype', values='pct_expr')

# Reorder TFs: Classical (negative triple score) first, then Special
tf_order = [t.replace('(+)', '') for t in top_cl_tfs] + \
           [t.replace('(+)', '') for t in top_sp_tfs]
tf_order = [t for t in tf_order if t in piv_z.index]
piv_z   = piv_z.loc[tf_order]
piv_pct = piv_pct.loc[tf_order]

celltypes_order = sorted(dot_df['celltype'].unique())
piv_z   = piv_z[celltypes_order]
piv_pct = piv_pct[celltypes_order]

fig, ax = plt.subplots(
    figsize=(len(celltypes_order) * 0.85 + 1.5, len(tf_order) * 0.55 + 1.2),
    dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

vmax = np.percentile(np.abs(piv_z.values[~np.isnan(piv_z.values)]), 95)
norm = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = mpl.cm.RdBu_r

for i, tf in enumerate(tf_order):
    for j, ct in enumerate(celltypes_order):
        z   = piv_z.loc[tf, ct]   if not np.isnan(piv_z.loc[tf, ct])   else 0
        pct = piv_pct.loc[tf, ct] if not np.isnan(piv_pct.loc[tf, ct]) else 0
        size = (pct / 100) * 300   # scale to dot area
        color = cmap(norm(z))
        ax.scatter(j, i, s=size, c=[color],
                   edgecolors='#555', linewidths=0.3, zorder=3)

# Separator line between Classical and Special TF groups
n_cl = len([t for t in top_cl_tfs if t.replace('(+)', '') in tf_order])
ax.axhline(n_cl - 0.5, color='#888', lw=1.0, ls='--', zorder=2)

# TF label color: Classical = blue, Special = red
ax.set_yticks(range(len(tf_order)))
ax.set_yticklabels(tf_order, fontsize=9)
for tick, tf in zip(ax.get_yticklabels(), tf_order):
    tick.set_color('#2471A3' if tf in [t.replace('(+)', '') for t in top_cl_tfs] else '#C0392B')

ax.set_xticks(range(len(celltypes_order)))
ax.set_xticklabels(celltypes_order, rotation=40, ha='right', fontsize=9)
ax.set_xlim(-0.5, len(celltypes_order) - 0.5)
ax.set_ylim(-0.5, len(tf_order) - 0.5)
ax.spines[['top', 'right']].set_visible(False)

# Colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=15)
cb.set_label('Z-score (AUC)', fontsize=9)
cb.ax.tick_params(labelsize=8)

# Size legend
from matplotlib.lines import Line2D
size_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
           markersize=np.sqrt(p/100*300), markeredgecolor='#555',
           markeredgewidth=0.3, label=f'{p}%')
    for p in [25, 50, 75]
]
ax.legend(handles=size_handles, title='% cells active',
          title_fontsize=8, frameon=False, fontsize=8,
          loc='upper left', bbox_to_anchor=(1.12, 1.0),
          handletextpad=0.2, labelspacing=0.5)

ax.set_title(
    'TF Regulon Activity across Cell Types\n'
    '(Blue label = Classical-specific  |  Red label = Special-specific)',
    fontsize=10, fontweight='bold', pad=10)

save_path2 = output_path('02_SLC7A5_mono_Journal/05_Figure/Figure_concordant_TF_dotplot.pdf')
plt.savefig(save_path2, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path2}")


# Z-score Figure — Classical vs Special monocyte
> **Z-score (group vs rest)** 만 사용한 분석: RSS 없이 각 subtype의 특이적 TF를 비교  
> **실행 순서**: 아래 Data Prep 셀 먼저 → Panel A → Panel B → Panel C 순서대로 실행
>
> 의존성: `auc_data` (cell 13), `adata.obs['celltype']` (cell 2)

In [ ]:
# ── Data Prep: Z-score (group vs rest) for Classical & Special monocyte ──
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

def compute_zscore_vs_rest(auc_df, celltype_series, group_name):
    """group_name vs all other cells: z-score (Welch t-stat) + FDR correction"""
    g1    = auc_df[celltype_series == group_name]
    g_oth = auc_df[celltype_series != group_name]
    rows  = []
    for tf in auc_df.columns:
        a, b = g1[tf].dropna(), g_oth[tf].dropna()
        stat, pval = ttest_ind(a, b, equal_var=False)
        se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
        z  = (a.mean() - b.mean()) / (se + 1e-12)
        rows.append({'TF': tf, 'z_score': z, 'pval': pval,
                     'mean_group': a.mean(), 'mean_rest': b.mean()})
    res = pd.DataFrame(rows).set_index('TF')
    res['padj'] = multipletests(res['pval'], method='fdr_bh')[1]
    res['sig']  = res['padj'] < 0.05
    return res

celltypes = adata.obs['celltype']
res_cl = compute_zscore_vs_rest(auc_data, celltypes, 'Classical_monocyte')
res_sp = compute_zscore_vs_rest(auc_data, celltypes, 'Special_monocyte')

zdf = pd.DataFrame({
    'z_classical': res_cl['z_score'],
    'z_special'  : res_sp['z_score'],
    'padj_cl'    : res_cl['padj'],
    'padj_sp'    : res_sp['padj'],
    'sig_cl'     : res_cl['sig'],
    'sig_sp'     : res_sp['sig'],
})
zdf['delta_z']  = zdf['z_special'] - zdf['z_classical']
zdf['TF_clean'] = zdf.index.str.replace('(+)', '', regex=False)

# ── Category assignment ───────────────────────────────────────
Z_CUTOFF = 5   # 의미있는 enrichment 최소 기준

both_negative = (zdf['z_classical'] < 0) & (zdf['z_special'] < 0)

zdf['category'] = 'Shared / NS'

# Classical-specific: sig AND z_cl >= cutoff AND z_cl > z_sp
zdf.loc[
    zdf['sig_cl'] &
    (zdf['z_classical'] >= Z_CUTOFF) &
    (zdf['z_classical'] > zdf['z_special']),
    'category'
] = 'Classical-specific'

# Special-specific: sig AND z_sp >= cutoff AND z_sp > z_cl
zdf.loc[
    zdf['sig_sp'] &
    (zdf['z_special'] >= Z_CUTOFF) &
    (zdf['z_special'] > zdf['z_classical']),
    'category'
] = 'Special-specific'

# Shared monocyte: 둘 다 sig AND 둘 다 z >= cutoff AND delta_z 작음
zdf.loc[
    zdf['sig_cl'] & zdf['sig_sp'] &
    (zdf['z_classical'] >= Z_CUTOFF) & (zdf['z_special'] >= Z_CUTOFF) &
    (zdf['delta_z'].abs() < 3),
    'category'
] = 'Shared monocyte'

print(f'Z_CUTOFF = {Z_CUTOFF}')
print('Category counts:')
print(zdf['category'].value_counts())
print(f'(Both z < 0, excluded: {both_negative.sum()} TFs)')
print()
print('=== Top 50 by delta_z (z_sp >= cutoff 필터 적용) ===')
top50 = (
    zdf[zdf['z_special'] >= Z_CUTOFF]
    .nlargest(50, 'delta_z')
    [['TF_clean', 'z_classical', 'z_special', 'delta_z', 'category']]
)
print(top50.to_string(index=False))


In [ ]:
# ── Panel A: Z-score scatter (z_Classical vs z_Special) ──────────────────
from adjustText import adjust_text
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 11,
                     'axes.linewidth': 0.8})

CLR   = '#2471A3'   # Classical
SLC   = '#C0392B'   # Special
SHR   = '#27AE60'   # Shared monocyte
GREY  = '#BDBDBD'   # NS / both negative

color_map = {
    'Classical-specific' : CLR,
    'Special-specific'   : SLC,
    'Shared monocyte'    : SHR,
    'Shared / NS'        : GREY,
}

fig, ax = plt.subplots(figsize=(6.5, 6.5), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.spines[['top', 'right']].set_visible(False)

# Plot by category (NS first so colored points sit on top)
for cat in ['Shared / NS', 'Shared monocyte', 'Classical-specific', 'Special-specific']:
    sub = zdf[zdf['category'] == cat]
    alpha = 0.20 if cat == 'Shared / NS' else 0.82
    size  = 15   if cat == 'Shared / NS' else 45
    ax.scatter(sub['z_classical'], sub['z_special'],
               c=color_map[cat], s=size, alpha=alpha,
               linewidths=0, label=cat, zorder=3)

# Diagonal reference line (y = x)
pos_max = zdf[['z_classical','z_special']].clip(lower=0).max().max() * 1.10
ax.plot([0, pos_max], [0, pos_max], '--', color='#888', lw=0.9, alpha=0.6, zorder=1)

# 사분면 구분선 (z = 0)
ax.axhline(0, color='#ccc', lw=0.7, zorder=1)
ax.axvline(0, color='#ccc', lw=0.7, zorder=1)

# 3사분면(둘 다 음수) 회색 배경으로 강조
xmin = zdf['z_classical'].min() * 1.05
ymin = zdf['z_special'].min() * 1.05
ax.fill_between([xmin, 0], [ymin, ymin], [0, 0],
                color='#F0F0F0', zorder=0, label='_nolegend_')
ax.text(xmin * 0.5, ymin * 0.5, 'Both z < 0\n(excluded)',
        ha='center', va='center', fontsize=8, color='#AAAAAA', style='italic')

xmax = zdf['z_classical'].max() * 1.05
ymax = zdf['z_special'].max() * 1.05
ax.set_xlim(-20, xmax)
ax.set_ylim(-20, ymax)

# Label top TFs per category
label_cl = zdf[zdf['category']=='Classical-specific'].nsmallest(8, 'delta_z')
label_sp = zdf[zdf['category']=='Special-specific'].nlargest(8, 'delta_z')
label_sh = zdf[zdf['category']=='Shared monocyte'].nlargest(5, 'z_classical')

texts = []
for _, row in pd.concat([label_cl, label_sp, label_sh]).iterrows():
    col = color_map[row['category']]
    texts.append(ax.text(row['z_classical'], row['z_special'],
                         row['TF_clean'], fontsize=8, color=col, fontweight='bold'))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

ax.set_xlabel('Z-score  (Classical vs rest)', fontsize=12)
ax.set_ylabel('Z-score  (Special vs rest)', fontsize=12)
ax.set_title('TF Regulon Activity — Z-score Comparison\n'
             'Classical vs Special monocyte (each vs all other cells)',
             fontsize=11, fontweight='bold', pad=10)
ax.legend(frameon=False, fontsize=9, loc='lower right',
          markerscale=1.2, handletextpad=0.3, labelspacing=0.4)

save_A = output_path('02_SLC7A5_mono_Journal/05_Figure/FigureZ_A_zscore_scatter.pdf')
plt.savefig(save_A, bbox_inches='tight')
plt.show()
print(f'Saved: {save_A}')


In [ ]:
# ── Panel B: Delta-z mirror barplot (z_Special - z_Classical) ────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 11,
                     'axes.linewidth': 0.8})

CLR = '#2471A3'
SLC = '#C0392B'

top_n    = 12
Z_CUTOFF = 5   # Data Prep과 동일

must_include = ['NFKB1(+)', 'NFKB2(+)']

# Special-enriched: z_special >= cutoff (Special이 실제로 높아야 함)
# Classical-enriched: z_classical >= cutoff (Classical이 실제로 높아야 함)
sp_pool = zdf[(zdf['sig_cl'] | zdf['sig_sp']) & (zdf['z_special'] >= Z_CUTOFF)]
cl_pool = zdf[(zdf['sig_cl'] | zdf['sig_sp']) & (zdf['z_classical'] >= Z_CUTOFF)]

top_sp_bar = sp_pool.nlargest(top_n, 'delta_z')
top_cl_bar = cl_pool.nsmallest(top_n, 'delta_z')

# Force-include
already = set(top_sp_bar.index) | set(top_cl_bar.index)
for tf in must_include:
    if tf in zdf.index and tf not in already:
        row = zdf.loc[[tf]]
        dz  = row['delta_z'].values[0]
        zsp = row['z_special'].values[0]
        zcl = row['z_classical'].values[0]
        if dz >= 0:
            top_sp_bar = pd.concat([top_sp_bar, row])
        else:
            top_cl_bar = pd.concat([top_cl_bar, row])
        print(f'Force-added: {tf}  z_cl={zcl:.2f}  z_sp={zsp:.2f}  delta_z={dz:.3f}')
    elif tf not in zdf.index:
        print(f'Warning: {tf} not found')

bar_data = pd.concat([top_cl_bar, top_sp_bar]).sort_values('delta_z')
bar_data = bar_data[~bar_data.index.duplicated()]

bar_colors = [CLR if v < 0 else SLC for v in bar_data['delta_z']]
is_forced  = bar_data.index.isin(must_include)

fig, ax = plt.subplots(figsize=(5.5, len(bar_data) * 0.38 + 1.2), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.spines[['top', 'right']].set_visible(False)

bars = ax.barh(range(len(bar_data)), bar_data['delta_z'],
               color=bar_colors, alpha=0.88, height=0.65, linewidth=0, zorder=3)

for bar, forced in zip(bars, is_forced):
    if forced:
        bar.set_edgecolor('black')
        bar.set_linewidth(1.4)

ax.set_yticks(range(len(bar_data)))
ax.set_yticklabels(bar_data['TF_clean'], fontsize=9.5)
for tick, forced in zip(ax.get_yticklabels(), is_forced):
    if forced:
        tick.set_fontweight('bold')
        tick.set_fontstyle('italic')

ax.axvline(0, color='black', lw=0.8)
ax.tick_params(axis='y', length=0)

val_range = bar_data['delta_z'].abs().max()
for bar, val, forced in zip(bars, bar_data['delta_z'], is_forced):
    offset = val_range * 0.03
    ax.text(val + offset if val >= 0 else val - offset,
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center', fontsize=7.5,
            ha='left' if val >= 0 else 'right',
            color='black' if forced else '#444',
            fontweight='bold' if forced else 'normal')

x_lim = ax.get_xlim()
ax.text(x_lim[0] * 0.92, len(bar_data) - 0.5,
        'Classical', ha='center', color=CLR, fontsize=9.5, fontweight='bold')
ax.text(x_lim[1] * 0.92, len(bar_data) - 0.5,
        'Special', ha='center', color=SLC, fontsize=9.5, fontweight='bold')

ax.set_xlabel('Δ Z-score  (Special − Classical)', fontsize=11)
ax.set_title(
    f'Top TFs by Differential Z-score  (z ≥ {Z_CUTOFF} cutoff)\n'
    '(positive = Special-enriched, negative = Classical-enriched)',
    fontsize=11, fontweight='bold', pad=10)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#888', edgecolor='black', linewidth=1.4,
                          label='Highlighted TF (forced)')],
          frameon=False, fontsize=8.5, loc='lower right')

save_B = output_path('02_SLC7A5_mono_Journal/05_Figure/FigureZ_B_deltaz_barplot.pdf')
plt.savefig(save_B, bbox_inches='tight')
plt.show()
print(f'Saved: {save_B}')


In [ ]:
# ── Grouped TF table: log2FC (Classical vs Special 직접 비교) ─────────────
# df = cell 52의 Mann-Whitney 결과 (index: 'TF(+)', columns: log2FC, padj, sig ...)
import pandas as pd
import numpy as np

tf_groups = {
    'FOXO axis'     : ['FOXO1', 'FOXO3', 'FOXC2', 'FOXU1'],
    'NF-κB complex' : ['NFKB1', 'NFKB2', 'RELB', 'RELA', 'BCL3'],
    'Metabolism'    : ['BHLHE40', 'HIF1A', 'ATF5', 'STAT3', 'STAT4'],
    'AP-1 resistant': ['FOSL1', 'JUND', 'CREM', 'MAFK'],
    'Epigenetic'    : ['TRIM28', 'MECP2', 'CHD2', 'BRCA1', 'EP300', 'ZBTB7A'],
}

# df index: 'TF(+)' → TF_clean 매핑
df_clean = df.copy()
df_clean['TF_clean'] = df_clean.index.str.replace('(+)', '', regex=False)
clean_to_row = df_clean.set_index('TF_clean')

rows = []
for group, tfs in tf_groups.items():
    for tf in tfs:
        if tf in clean_to_row.index:
            r = clean_to_row.loc[tf]
            rows.append({
                'Group'  : group,
                'TF'     : tf,
                'log2FC' : round(r['log2FC'], 4),     # + = Special 높음
                'padj'   : round(r['padj'], 6),
                'sig'    : r['sig'],
                'mean_Classical'  : round(r['mean_Classical'], 6),
                'mean_SLC7A5_high': round(r['mean_SLC7A5_high'], 6),
            })
        else:
            print(f'Not found in Mann-Whitney results: {tf}')
            rows.append({
                'Group': group, 'TF': tf,
                'log2FC': np.nan, 'padj': np.nan, 'sig': False,
                'mean_Classical': np.nan, 'mean_SLC7A5_high': np.nan,
            })

group_df = pd.DataFrame(rows)
group_df['direction'] = group_df['log2FC'].apply(
    lambda x: 'Special-enriched' if x > 0 else ('Classical-enriched' if x < 0 else '-')
)

print('=== Grouped TF — log2FC (Special / Classical, Mann-Whitney) ===')
print(f'log2FC > 0 : Special monocyte 높음')
print(f'log2FC < 0 : Classical monocyte 높음')
print()
print(group_df.to_string(index=False))
print()
print('=== Summary by group ===')
print(group_df.groupby('Group')[['log2FC','sig']].agg(
    mean_log2FC=('log2FC','mean'),
    n_sig=('sig','sum'),
    n_total=('sig','count')
).to_string())


In [ ]:
# ── Significant TF only — Grouped barplot (log2FC, Classical vs Special) ──
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8})

group_colors = {
    'FOXO axis'     : '#8E44AD',
    'NF-κB complex' : '#C0392B',
    'Metabolism'    : '#E67E22',
    'AP-1 resistant': '#27AE60',
    'Epigenetic'    : '#2471A3',
}
group_order = list(group_colors.keys())

# sig TF만 필터, 그룹 순서 고정 후 그룹 내 log2FC 내림차순
sig_df = group_df[group_df['sig']].copy()
sig_df['Group'] = pd.Categorical(sig_df['Group'], categories=group_order, ordered=True)
sig_df = sig_df.sort_values(['Group', 'log2FC'], ascending=[True, False]).reset_index(drop=True)

if sig_df.empty:
    print('No significant TFs found — check padj cutoff or TF names')
else:
    print(f'Significant TFs: {len(sig_df)}')
    print(sig_df[['Group','TF','log2FC','padj']].to_string(index=False))

    fig, ax = plt.subplots(figsize=(5.5, len(sig_df) * 0.46 + 1.8), dpi=200)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    ypos       = np.arange(len(sig_df))
    bar_colors = [group_colors[g] for g in sig_df['Group']]

    bars = ax.barh(ypos, sig_df['log2FC'],
                   color=bar_colors, alpha=0.88, height=0.65,
                   linewidth=0, zorder=3)

    # log2FC 값 라벨
    val_range = sig_df['log2FC'].abs().max()
    for bar, val in zip(bars, sig_df['log2FC']):
        offset = val_range * 0.03
        ax.text(val + offset if val >= 0 else val - offset,
                bar.get_y() + bar.get_height() / 2,
                f'{val:+.3f}', va='center', fontsize=7.5,
                ha='left' if val >= 0 else 'right', color='#333')

    # Y tick labels
    ax.set_yticks(ypos)
    ax.set_yticklabels(sig_df['TF'], fontsize=10)
    for tick, color in zip(ax.get_yticklabels(), bar_colors):
        tick.set_color(color)
        tick.set_fontweight('bold')

    # 그룹 구분선 + 오른쪽 브라켓 + 그룹명
    prev_end = -1
    for group in group_order:
        idxs = sig_df[sig_df['Group'] == group].index.tolist()
        if not idxs:
            continue
        start, end = min(idxs), max(idxs)
        if prev_end >= 0:
            ax.axhline(prev_end + 0.5, color='#DDDDDD', lw=0.8, zorder=1)
        # 브라켓
        ax.annotate('', xy=(1.01, start / len(sig_df)),
                    xycoords=('axes fraction', 'axes fraction'),
                    xytext=(1.01, (end + 1) / len(sig_df)),
                    textcoords=('axes fraction', 'axes fraction'),
                    arrowprops=dict(arrowstyle='-', color=group_colors[group],
                                    lw=2.0), annotation_clip=False)
        # 그룹명
        ax.annotate(
            group,
            xy=(1.025, (start + end + 1) / (2 * len(sig_df))),
            xycoords=('axes fraction', 'axes fraction'),
            fontsize=8.5, color=group_colors[group], fontweight='bold',
            va='center', ha='left', annotation_clip=False
        )
        prev_end = end

    ax.axvline(0, color='black', lw=0.8)
    ax.tick_params(axis='y', length=0)

    # x축 방향 표시
    xlim = ax.get_xlim()
    ax.text(xlim[0], -1.5, '← Classical',
            ha='left', fontsize=8.5, color='#2471A3', style='italic')
    ax.text(xlim[1], -1.5, 'Special →',
            ha='right', fontsize=8.5, color='#C0392B', style='italic')

    ax.set_xlabel('log2FC  (Special / Classical monocyte)', fontsize=11)
    ax.set_title(
        'Regulon Activity of Functional TF Groups\n'
        '(significant TFs only, FDR < 0.05)',
        fontsize=11, fontweight='bold', pad=10)

    plt.tight_layout(rect=[0, 0, 0.82, 1])   # 오른쪽 브라켓 공간 확보
    save_path = output_path('02_SLC7A5_mono_Journal/05_Figure/Figure3B_sig_grouped_barplot.pdf')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


In [ ]:
# ── Panel B-2: Grouped barplot by functional TF category ─────────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8})

# ── TF 그룹 정의 ─────────────────────────────────────────────
tf_groups = {
    'FOXO axis'     : ['FOXO1', 'FOXO3', 'FOXC2', 'FOXU1'],
    'NF-κB complex' : ['NFKB1', 'NFKB2', 'RELB', 'RELA', 'BCL3'],
    'Metabolism'    : ['BHLHE40', 'HIF1A', 'ATF5', 'STAT3', 'STAT4'],
    'AP-1 resistant': ['FOSL1', 'JUND', 'CREM', 'MAFK'],
    'Epigenetic'    : ['TRIM28', 'MECP2', 'CHD2', 'BRCA1', 'EP300', 'ZBTB7A'],
}

# 그룹별 색상
group_colors = {
    'FOXO axis'     : '#8E44AD',
    'NF-κB complex' : '#C0392B',
    'Metabolism'    : '#E67E22',
    'AP-1 resistant': '#27AE60',
    'Epigenetic'    : '#2471A3',
}

# ── delta_z 값 매핑 ──────────────────────────────────────────
# zdf index는 'TF(+)' 형태, TF_clean은 '(+)' 제거된 것
clean_to_idx = dict(zip(zdf['TF_clean'], zdf.index))

rows = []
for group, tfs in tf_groups.items():
    for tf in tfs:
        if tf in clean_to_idx:
            idx = clean_to_idx[tf]
            rows.append({
                'TF'     : tf,
                'group'  : group,
                'delta_z': zdf.loc[idx, 'delta_z'],
                'z_sp'   : zdf.loc[idx, 'z_special'],
                'z_cl'   : zdf.loc[idx, 'z_classical'],
                'sig'    : zdf.loc[idx, 'sig_cl'] or zdf.loc[idx, 'sig_sp'],
            })
        else:
            print(f'Not found in zdf: {tf}')
            rows.append({
                'TF': tf, 'group': group,
                'delta_z': np.nan, 'z_sp': np.nan, 'z_cl': np.nan, 'sig': False
            })

gdf = pd.DataFrame(rows)

# 그룹 내 delta_z 내림차순 정렬
gdf = gdf.sort_values(['group', 'delta_z'], ascending=[True, False])
# 그룹 순서 고정
group_order = list(tf_groups.keys())
gdf['group'] = pd.Categorical(gdf['group'], categories=group_order, ordered=True)
gdf = gdf.sort_values(['group', 'delta_z'], ascending=[True, False]).reset_index(drop=True)

# ── Plot ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, len(gdf) * 0.42 + 2.0), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.spines[['top', 'right']].set_visible(False)

ypos = np.arange(len(gdf))
bar_colors = [group_colors[g] for g in gdf['group']]
alphas     = [0.90 if s else 0.35 for s in gdf['sig']]

bars = ax.barh(ypos, gdf['delta_z'],
               color=bar_colors, alpha=0.88, height=0.65, linewidth=0, zorder=3)

# 비유의미 TF: 점선 테두리
for bar, sig in zip(bars, gdf['sig']):
    if not sig:
        bar.set_edgecolor('#888')
        bar.set_linewidth(0.8)
        bar.set_linestyle('--')
        bar.set_alpha(0.40)

# Y tick labels
ax.set_yticks(ypos)
ax.set_yticklabels(gdf['TF'], fontsize=9.5)
for tick, sig in zip(ax.get_yticklabels(), gdf['sig']):
    if not sig:
        tick.set_color('#AAAAAA')

# delta_z 값 라벨
val_range = gdf['delta_z'].abs().max()
for bar, val, sig in zip(bars, gdf['delta_z'], gdf['sig']):
    if np.isnan(val):
        continue
    offset = val_range * 0.03
    ax.text(val + offset if val >= 0 else val - offset,
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center', fontsize=7,
            ha='left' if val >= 0 else 'right',
            color='#333' if sig else '#AAAAAA')

# 그룹 구분선 + 그룹명 라벨
x_min = ax.get_xlim()[0]
prev_end = -1
for group in group_order:
    idxs = gdf[gdf['group'] == group].index.tolist()
    if not idxs:
        continue
    start, end = min(idxs), max(idxs)
    mid = (start + end) / 2
    # 그룹 구분선 (첫 그룹 제외)
    if prev_end >= 0:
        ax.axhline(prev_end + 0.5, color='#CCCCCC', lw=0.8, zorder=1)
    # 그룹명: 오른쪽 여백에 브라켓
    ax.annotate(
        group,
        xy=(1.01, (start + end + 1) / (2 * len(gdf))),
        xycoords=('axes fraction', 'axes fraction'),
        fontsize=8.5, color=group_colors[group], fontweight='bold',
        va='center', ha='left',
        annotation_clip=False
    )
    # 세로 브라켓 선
    ax.annotate('', xy=(1.005, start / len(gdf)),
                xycoords=('axes fraction', 'axes fraction'),
                xytext=(1.005, (end + 1) / len(gdf)),
                textcoords=('axes fraction', 'axes fraction'),
                arrowprops=dict(arrowstyle='-', color=group_colors[group],
                                lw=1.8), annotation_clip=False)
    prev_end = end

ax.axvline(0, color='black', lw=0.8)
ax.tick_params(axis='y', length=0)

# x축 방향 레이블
xlim = ax.get_xlim()
ax.text(xlim[0], -1.2, '← Classical-enriched',
        ha='left', fontsize=8.5, color='#2471A3', style='italic')
ax.text(xlim[1], -1.2, 'Special-enriched →',
        ha='right', fontsize=8.5, color='#C0392B', style='italic')

ax.set_xlabel('Δ Z-score  (Special − Classical)', fontsize=11)
ax.set_title('TF Functional Group — Differential Activity\n'
             '(Special vs Classical monocyte, faded = NS)',
             fontsize=11, fontweight='bold', pad=10)

plt.tight_layout()
save_Bg = output_path('02_SLC7A5_mono_Journal/05_Figure/Figure3B_grouped_barplotv2.pdf')
plt.savefig(save_Bg, bbox_inches='tight')
plt.show()
print(f'Saved: {save_Bg}')
print()
print(gdf[['group','TF','z_cl','z_sp','delta_z','sig']].to_string(index=False))


In [ ]:
# ── Panel C: AUC dot plot — selected TFs × all cell types ────────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.size': 10,
                     'axes.linewidth': 0.8})

CLR = '#2471A3'
SLC = '#C0392B'

top_n_sel = 8   # per group

# Select top TFs: classical-specific (most negative delta_z) + special-specific (most positive)
zdf_sig = zdf[zdf['sig_cl'] | zdf['sig_sp']].copy()
top_cl_tfs = zdf_sig.nsmallest(top_n_sel, 'delta_z').index.tolist()
top_sp_tfs = zdf_sig.nlargest(top_n_sel,  'delta_z').index.tolist()
sel_tfs    = top_cl_tfs + top_sp_tfs   # Classical on top, Special on bottom

# Mean AUC and pct-expressing per (TF, celltype)
ct_series = adata.obs['celltype']
rows_dot  = []
for ct in sorted(ct_series.unique()):
    mask = ct_series == ct
    for tf in sel_tfs:
        vals = auc_data.loc[mask, tf].dropna()
        if len(vals) == 0:
            continue
        q25 = vals.quantile(0.25)
        rows_dot.append({
            'celltype' : ct,
            'TF'       : tf,
            'TF_clean' : tf.replace('(+)', ''),
            'mean_auc' : vals.mean(),
            'pct_expr' : (vals > q25).mean() * 100
        })

dot_df = pd.DataFrame(rows_dot)

# Z-score normalise mean_auc per TF across cell types
dot_df['z_auc'] = dot_df.groupby('TF')['mean_auc'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-10))

# Pivot
piv_z   = dot_df.pivot(index='TF', columns='celltype', values='z_auc')
piv_pct = dot_df.pivot(index='TF', columns='celltype', values='pct_expr')

# Row order: Classical group first (most negative delta_z at top), then Special
tf_order = top_cl_tfs + top_sp_tfs
tf_order = [t for t in tf_order if t in piv_z.index]
piv_z    = piv_z.loc[tf_order]
piv_pct  = piv_pct.loc[tf_order]

# Column order: put Classical & Special monocyte adjacent
ct_all = sorted(ct_series.unique())
priority = ['Classical_monocyte', 'Special_monocyte']
ct_order = priority + [c for c in ct_all if c not in priority]
ct_order = [c for c in ct_order if c in piv_z.columns]
piv_z    = piv_z[ct_order]
piv_pct  = piv_pct[ct_order]

# ── Draw ──
fig, ax = plt.subplots(
    figsize=(len(ct_order) * 0.9 + 1.8, len(tf_order) * 0.52 + 1.5),
    dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

vmax  = np.percentile(np.abs(piv_z.values[~np.isnan(piv_z.values)]), 95)
norm  = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap  = mpl.cm.RdBu_r

for i, tf in enumerate(tf_order):
    for j, ct in enumerate(ct_order):
        z_val   = piv_z.loc[tf, ct]   if not np.isnan(piv_z.loc[tf, ct])   else 0
        pct_val = piv_pct.loc[tf, ct] if not np.isnan(piv_pct.loc[tf, ct]) else 0
        size    = (pct_val / 100) * 320
        ax.scatter(j, i, s=size, c=[cmap(norm(z_val))],
                   edgecolors='#666', linewidths=0.25, zorder=3)

# Separator: Classical vs Special TF groups
n_cl_tfs = len([t for t in top_cl_tfs if t in tf_order])
ax.axhline(n_cl_tfs - 0.5, color='#888', lw=1.0, ls='--', zorder=2)

# Separator: monocyte vs other cell types (after 2nd column)
ax.axvline(len(priority) - 0.5, color='#888', lw=1.0, ls='--', zorder=2)

# Y-axis: TF labels colored by group
tf_clean_order = [t.replace('(+)', '') for t in tf_order]
ax.set_yticks(range(len(tf_order)))
ax.set_yticklabels(tf_clean_order, fontsize=9.5)
cl_clean = [t.replace('(+)', '') for t in top_cl_tfs]
for tick, tf_c in zip(ax.get_yticklabels(), tf_clean_order):
    tick.set_color(CLR if tf_c in cl_clean else SLC)
    tick.set_fontweight('bold')

# X-axis: cell type labels
ax.set_xticks(range(len(ct_order)))
ax.set_xticklabels(ct_order, rotation=40, ha='right', fontsize=9)
ax.set_xlim(-0.5, len(ct_order) - 0.5)
ax.set_ylim(-0.5, len(tf_order) - 0.5)
ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)
ax.tick_params(length=0)

# Colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, shrink=0.35, pad=0.02, aspect=14)
cb.set_label('Z-score (AUC)', fontsize=9)
cb.ax.tick_params(labelsize=8)

# Size legend
from matplotlib.lines import Line2D
size_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
           markersize=np.sqrt(p/100*320), markeredgecolor='#666',
           markeredgewidth=0.3, label=f'{p}%')
    for p in [25, 50, 75]
]
ax.legend(handles=size_handles, title='% cells\nactive',
          title_fontsize=8, frameon=False, fontsize=8.5,
          loc='upper left', bbox_to_anchor=(1.12, 1.02),
          handletextpad=0.2, labelspacing=0.6)

ax.set_title(
    'TF Regulon Activity across Cell Types\n'
    '(Blue = Classical-specific  |  Red = Special-specific)',
    fontsize=10, fontweight='bold', pad=10)

save_C = output_path('02_SLC7A5_mono_Journal/05_Figure/Figure3B_auc_dotplot.pdf')
plt.savefig(save_C, bbox_inches='tight')
plt.show()
print(f'Saved: {save_C}')


# Special monocyte — Regulon Analysis & Pathway Enrichment
> **Special-specific TFs**: CHD2, ZBTB7A, MECP2, TRIM28, JUND, FOSL1, MAFK, BHLHE40, STAT4, ATF5, RELB, NFKB1, NFKB2, BCL3, FOXC2, FOXO1  
> **Cell 순서**: Data Prep → Enrichment → Dot plot → Lollipop → Network
> 의존성: `regulons` (cell 12), `adj` TSV (cell 59), `auc_data` (cell 13)

In [ ]:
# ── Regulon Data Prep: Special-specific TF target genes 정리 ─────────────
import pandas as pd
import numpy as np
from itertools import combinations

SPECIAL_TFS = [
    'CHD2', 'ZBTB7A', 'MECP2', 'TRIM28', 'JUND', 'FOSL1', 'MAFK',
    'BHLHE40', 'STAT4', 'ATF5', 'RELB', 'NFKB1', 'NFKB2', 'BCL3',
    'FOXC2', 'FOXO1'
]

TF_GROUPS = {
    'FOXO axis'     : ['FOXO1', 'FOXC2'],
    'NF-κB complex' : ['NFKB1', 'NFKB2', 'RELB', 'BCL3'],
    'Metabolism'    : ['BHLHE40', 'ATF5', 'STAT4'],
    'AP-1 resistant': ['FOSL1', 'JUND', 'MAFK'],
    'Epigenetic'    : ['TRIM28', 'MECP2', 'CHD2', 'ZBTB7A'],
}

GROUP_COLORS = {
    'FOXO axis'     : '#8E44AD',
    'NF-κB complex' : '#C0392B',
    'Metabolism'    : '#E67E22',
    'AP-1 resistant': '#27AE60',
    'Epigenetic'    : '#2471A3',
}

# TF → group 역매핑
tf_to_group = {tf: g for g, tfs in TF_GROUPS.items() for tf in tfs}

# regulons에서 target genes 추출
# regulons key: 'TF(+)' 또는 'TF' 형태 확인
reg_keys = list(regulons.keys())[:5]
print('regulons key 예시:', reg_keys)

regulon_sp = {}   # TF → target gene list
missing = []
for tf in SPECIAL_TFS:
    key_plus = tf + '(+)'
    if tf in regulons:
        regulon_sp[tf] = list(set(regulons[tf]))
    elif key_plus in regulons:
        regulon_sp[tf] = list(set(regulons[key_plus]))
    else:
        missing.append(tf)
        regulon_sp[tf] = []

if missing:
    print(f'regulons에 없는 TF: {missing}')

# Summary table
summary_rows = []
for tf in SPECIAL_TFS:
    targets = regulon_sp[tf]
    summary_rows.append({
        'TF'      : tf,
        'Group'   : tf_to_group.get(tf, 'Unknown'),
        'n_targets': len(targets),
    })
regulon_summary = pd.DataFrame(summary_rows)
print()
print('=== Regulon Summary ===')
print(regulon_summary.sort_values('n_targets', ascending=False).to_string(index=False))

# 전체 union target genes
all_targets = list(set(sum(regulon_sp.values(), [])))
print(f'\nTotal unique target genes across all Special TFs: {len(all_targets)}')

# 그룹별 target gene union
group_targets = {}
for group, tfs in TF_GROUPS.items():
    genes = list(set(sum([regulon_sp.get(tf, []) for tf in tfs], [])))
    group_targets[group] = genes
    print(f'  {group}: {len(genes)} unique targets')


In [ ]:
regulon_sp

In [ ]:
# ── GO BP & KEGG Enrichment — per group + all combined ───────────────────
import gseapy as gp
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def run_enrichr(gene_list, label, gene_sets=None, top_n=10):
    if gene_sets is None:
        gene_sets = ['GO_Biological_Process_2023', 'KEGG_2021_Human']
    if len(gene_list) == 0:
        print(f'[{label}] No genes')
        return pd.DataFrame()
    try:
        enr = gp.enrichr(gene_list=gene_list, gene_sets=gene_sets,
                         organism='human', outdir=None, verbose=False)
        res = enr.results.copy()
        res = res[res['Adjusted P-value'] < 0.05].copy()
        res['-log10padj'] = -np.log10(res['Adjusted P-value'].clip(lower=1e-300))
        res['GeneSet'] = label
        print(f'[{label}] sig terms: {len(res)}')
        return res
    except Exception as e:
        print(f'[{label}] Error: {e}')
        return pd.DataFrame()

# 전체 target genes enrichment
print('=== All Special TF targets ===')
enr_all = run_enrichr(all_targets, 'All_Special')

# 그룹별 enrichment
print()
print('=== Per-group enrichment ===')
enr_group = {}
for group, genes in group_targets.items():
    enr_group[group] = run_enrichr(genes, group)

# KEGG만 추출
def filter_kegg(df, top_n=10):
    if df.empty:
        return df
    k = df[df['Gene_set'] == 'KEGG_2021_Human'].nsmallest(top_n, 'Adjusted P-value')
    return k

def filter_go(df, top_n=10):
    if df.empty:
        return df
    g = df[df['Gene_set'] == 'GO_Biological_Process_2023'].nsmallest(top_n, 'Adjusted P-value')
    return g

kegg_all = filter_kegg(enr_all)
go_all   = filter_go(enr_all)

print()
print('=== Top KEGG (all targets) ===')
if not kegg_all.empty:
    print(kegg_all[['Term','Overlap','Adjusted P-value']].to_string(index=False))
print()
print('=== Top GO BP (all targets) ===')
if not go_all.empty:
    print(go_all[['Term','Overlap','Adjusted P-value']].to_string(index=False))


In [ ]:
# ── Enrichment Dot plot — per group, KEGG top 5 each ─────────────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8})

top_n_terms = 5

def shorten_term(name, n=45):
    s = name.split(' (GO:')[0].split(' KEGG')[0].split(' Homo')[0]
    return s[:n] + '..' if len(s) > n else s

# 그룹별 KEGG top-5 수집
rows_dot = []
for group in list(TF_GROUPS.keys())[::-1]:   # 아래→위 순서
    df_g = enr_group.get(group, pd.DataFrame())
    if df_g.empty:
        continue
    kegg_g = df_g[df_g['Gene_set'] == 'KEGG_2021_Human'].nsmallest(top_n_terms, 'Adjusted P-value')
    for _, row in kegg_g.iterrows():
        overlap_n = int(row['Overlap'].split('/')[0])
        total_n   = int(row['Overlap'].split('/')[1])
        rows_dot.append({
            'Group'     : group,
            'Term'      : shorten_term(row['Term']),
            'log10padj' : row['-log10padj'],
            'GeneRatio' : overlap_n / total_n,
            'Count'     : overlap_n,
        })

if not rows_dot:
    print('No significant KEGG terms found — try GO_Biological_Process_2023')
else:
    dot_df = pd.DataFrame(rows_dot)

    fig, ax = plt.subplots(figsize=(8, len(dot_df) * 0.42 + 1.5), dpi=200)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    norm  = mpl.colors.Normalize(vmin=dot_df['log10padj'].min(),
                                  vmax=dot_df['log10padj'].max())
    cmap  = mpl.cm.YlOrRd
    sizes = (dot_df['GeneRatio'] / dot_df['GeneRatio'].max()) * 300 + 30

    ypos = np.arange(len(dot_df))
    sc = ax.scatter(dot_df['GeneRatio'], ypos,
                    s=sizes, c=dot_df['log10padj'],
                    cmap=cmap, norm=norm,
                    edgecolors='#555', linewidths=0.3, zorder=3)

    ax.set_yticks(ypos)
    ax.set_yticklabels(dot_df['Term'], fontsize=8.5)

    # 그룹 구분선 + 오른쪽 브라켓
    prev_end = -1
    for group in list(TF_GROUPS.keys())[::-1]:
        idxs = dot_df[dot_df['Group'] == group].index.tolist()
        if not idxs:
            continue
        start, end = min(idxs), max(idxs)
        if prev_end >= 0:
            ax.axhline(prev_end + 0.5, color='#DDDDDD', lw=0.8, zorder=1)
        ax.annotate('', xy=(1.01, start / len(dot_df)),
                    xycoords=('axes fraction', 'axes fraction'),
                    xytext=(1.01, (end + 1) / len(dot_df)),
                    textcoords=('axes fraction', 'axes fraction'),
                    arrowprops=dict(arrowstyle='-', color=GROUP_COLORS[group],
                                    lw=2.0), annotation_clip=False)
        ax.annotate(group,
                    xy=(1.025, (start + end + 1) / (2 * len(dot_df))),
                    xycoords=('axes fraction', 'axes fraction'),
                    fontsize=8, color=GROUP_COLORS[group], fontweight='bold',
                    va='center', ha='left', annotation_clip=False)
        prev_end = end

    cb = fig.colorbar(sc, ax=ax, shrink=0.35, pad=0.18, aspect=14)
    cb.set_label('−log₁₀(FDR)', fontsize=9)
    cb.ax.tick_params(labelsize=8)

    # Size legend
    from matplotlib.lines import Line2D
    ratios = [0.1, 0.2, 0.3]
    size_leg = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
               markersize=np.sqrt((r / dot_df['GeneRatio'].max()) * 300 + 30),
               markeredgecolor='#555', markeredgewidth=0.3, label=f'{r:.1f}')
        for r in ratios
    ]
    ax.legend(handles=size_leg, title='Gene Ratio', title_fontsize=8,
              frameon=False, fontsize=8, loc='lower right',
              bbox_to_anchor=(1.0, 0.0), labelspacing=0.5)

    ax.set_xlabel('Gene Ratio', fontsize=11)
    ax.set_title('KEGG Pathway Enrichment\nSpecial monocyte Regulon Targets (per TF group)',
                 fontsize=11, fontweight='bold', pad=10)
    plt.tight_layout(rect=[0, 0, 0.80, 1])

    save_enr = output_path('02_SLC7A5_mono_Journal/05_Figure/FigureR_enrichment_dotplot.pdf')
    plt.savefig(save_enr, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_enr}')


In [ ]:
# ── KEGG Lollipop — all Special TF targets combined, top 15 ──────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 10,
                     'axes.linewidth': 0.8})

def shorten_term(name, n=50):
    s = name.split(' (GO:')[0].split(' KEGG')[0].split(' Homo')[0]
    return s[:n] + '..' if len(s) > n else s

for db_name, db_key in [('KEGG', 'KEGG_2021_Human'), ('GO BP', 'GO_Biological_Process_2023')]:
    if enr_all.empty:
        print(f'No enrichment results for {db_name}')
        continue

    sub = enr_all[enr_all['Gene_set'] == db_key].nsmallest(15, 'Adjusted P-value').copy()
    if sub.empty:
        print(f'No significant {db_name} terms')
        continue

    sub['Term_short'] = sub['Term'].apply(shorten_term)
    sub['Count'] = sub['Overlap'].apply(lambda x: int(x.split('/')[0]))
    sub = sub.sort_values('-log10padj', ascending=True)

    fig, ax = plt.subplots(figsize=(7, len(sub) * 0.45 + 1.2), dpi=200)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

    color = '#C0392B' if db_name == 'KEGG' else '#2471A3'
    ypos  = np.arange(len(sub))

    ax.hlines(ypos, 0, sub['-log10padj'].values,
              color=color, alpha=0.5, linewidth=1.5)
    sc = ax.scatter(sub['-log10padj'], ypos,
                    s=sub['Count'] * 8 + 20,
                    c=sub['-log10padj'], cmap='YlOrRd',
                    edgecolors='#555', linewidths=0.3, zorder=3)

    ax.set_yticks(ypos)
    ax.set_yticklabels(sub['Term_short'], fontsize=8.5)
    ax.set_xlabel('−log₁₀(FDR)', fontsize=11)
    ax.set_title(f'{db_name} Enrichment — All Special monocyte Regulon Targets\n'
                 f'(n = {len(all_targets)} unique genes)',
                 fontsize=11, fontweight='bold', pad=10)
    ax.tick_params(axis='y', length=0)

    cb = fig.colorbar(sc, ax=ax, shrink=0.35, pad=0.02, aspect=14)
    cb.set_label('−log₁₀(FDR)', fontsize=9)
    cb.ax.tick_params(labelsize=8)

    plt.tight_layout()
    save_lol = output_path(f'02_SLC7A5_mono_Journal/05_Figure/FigureR_{db_name.replace(" ","_")}_lollipop.pdf')
    plt.savefig(save_lol, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_lol}')


In [ ]:
# ── Regulon Network — Special TFs → top target genes ─────────────────────
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

mpl.rcParams.update({'font.family': 'Liberation Sans', 'font.size': 9,
                     'axes.linewidth': 0.8})

# adj TSV에서 Special TF만 필터
adj = pd.read_csv(
    artifact_path('02_SLC7A5_mono_Journal/02_Analysis/pySCENIC/data/cell_mono_SCENIC.adjacencies.tsv'),
    sep='\t'
)
print('adj columns:', adj.columns.tolist())

# importance 기준 상위 target만 사용 (너무 많으면 지저분)
top_per_tf = 10
adj_sp = (
    adj[adj['TF'].isin(SPECIAL_TFS)]
    .sort_values('importance', ascending=False)
    .groupby('TF')
    .head(top_per_tf)
    .reset_index(drop=True)
)

G = nx.DiGraph()
for _, row in adj_sp.iterrows():
    G.add_edge(row['TF'], row['target'], weight=row['importance'])

# 노드 속성 지정
tf_nodes     = [n for n in G.nodes() if n in SPECIAL_TFS]
target_nodes = [n for n in G.nodes() if n not in SPECIAL_TFS]

# target 중 여러 TF에 의해 공유되는 hub gene 찾기
in_degree  = dict(G.in_degree())
hub_genes  = {n for n, d in in_degree.items() if d >= 3 and n in target_nodes}

print(f'TF nodes: {len(tf_nodes)}, Target nodes: {len(target_nodes)}')
print(f'Hub genes (targeted by ≥3 TFs): {sorted(hub_genes)}')

# Layout: TF는 원형 바깥, target은 내부
pos_tf  = nx.circular_layout(nx.subgraph(G, tf_nodes), scale=3.0)
pos_tgt = nx.spring_layout(nx.subgraph(G, target_nodes), seed=42, scale=1.8)
pos = {**pos_tf, **pos_tgt}

fig, ax = plt.subplots(figsize=(14, 12), dpi=200)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.axis('off')

# TF 노드 색상: group별
tf_node_colors = [GROUP_COLORS.get(tf_to_group.get(n, ''), '#888888') for n in tf_nodes]

# edges
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_alphas = [0.15 + 0.55 * (w / max_w) for w in edge_weights]

nx.draw_networkx_edges(G, pos, ax=ax,
                       edge_color=['#AAAAAA'] * len(G.edges()),
                       alpha=edge_alphas, width=0.6,
                       arrows=True, arrowsize=8,
                       connectionstyle='arc3,rad=0.1')

# target nodes
hub_colors   = ['#F39C12' if n in hub_genes else '#D5D8DC' for n in target_nodes]
hub_sizes    = [350 if n in hub_genes else 80 for n in target_nodes]
nx.draw_networkx_nodes(G, pos, nodelist=target_nodes, ax=ax,
                       node_color=hub_colors, node_size=hub_sizes,
                       alpha=0.75, linewidths=0)

# TF nodes (위에 그려서 앞에 보이게)
nx.draw_networkx_nodes(G, pos, nodelist=tf_nodes, ax=ax,
                       node_color=tf_node_colors, node_size=600,
                       alpha=0.95, linewidths=0.8,
                       edgecolors='black')

# Labels: TF는 항상, target은 hub만
tf_labels  = {n: n for n in tf_nodes}
hub_labels = {n: n for n in hub_genes}
nx.draw_networkx_labels(G, pos, tf_labels, ax=ax,
                        font_size=8.5, font_weight='bold')
nx.draw_networkx_labels(G, pos, hub_labels, ax=ax,
                        font_size=7, font_color='#7D6608')

# Legend
legend_handles = [
    mpatches.Patch(facecolor=c, label=g, edgecolor='black', linewidth=0.5)
    for g, c in GROUP_COLORS.items()
] + [
    mpatches.Patch(facecolor='#F39C12', label='Hub gene (≥3 TFs)', edgecolor='none'),
    mpatches.Patch(facecolor='#D5D8DC', label='Target gene', edgecolor='none'),
]
ax.legend(handles=legend_handles, frameon=False, fontsize=8.5,
          loc='lower left', bbox_to_anchor=(0.0, 0.0),
          title='TF group', title_fontsize=9, labelspacing=0.4)

ax.set_title('Special monocyte Regulon Network\n'
             f'(top {top_per_tf} targets per TF by importance)',
             fontsize=12, fontweight='bold', pad=12)

save_net = output_path('02_SLC7A5_mono_Journal/05_Figure/FigureR_regulon_network.pdf')
plt.savefig(save_net, bbox_inches='tight')
plt.show()
print(f'Saved: {save_net}')
